# Этап 2. NLP-инструменты: лемматизация и токенизация

В этом ноутбуке реализованы два подхода к лемматизации македонских текстов и проведено их сравнение.

**Лемматизация** — приведение слова к начальной (словарной) форме. Например: *мачката* -> *мачка*, *читаше* -> *чита*, *големата* -> *голем*. Это ключевой шаг для NLP: без лемматизации слова *книга*, *книгата*, *книгите* считаются разными словами, хотя означают одно и то же.

**Два подхода к лемматизации:**
1. **Нейронный (CLASSLA)** — нейросеть, обученная на размеченных корпусах, которая предсказывает лемму по контексту
2. **Rule-based (собственный)** — набор правил на основе морфологии македонского языка: словарь исключений + отсечение суффиксов по POS-тегу

**Что делаем в этом ноутбуке:**
- Настраиваем CLASSLA для македонского языка и тестируем его
- Описываем морфологию македонского, нужную для rule-based лемматизатора
- Реализуем собственный rule-based лемматизатор
- Обрабатываем весь корпус (98 текстов) обоими методами
- Загружаем gold standard (UD Macedonian-MTB) для объективной оценки
- Сравниваем accuracy, скорость и ошибки двух подходов
- Настраиваем dependency parsing через spaCy
- Выбираем итоговый набор NLP-инструментов для проекта

## 2.1. CLASSLA — нейронный NLP-инструмент для южнославянских языков

**CLASSLA** — форк библиотеки Stanza (Stanford NLP), специально доработанный для южнославянских языков: словенский, хорватский, сербский, боснийский, македонский, болгарский.

Для македонского языка CLASSLA поддерживает:
- **tokenize** — разбиение текста на токены и предложения
- **pos** — POS-тегирование по стандарту Universal Dependencies
- **lemma** — лемматизация (приведение слова к начальной форме)

CLASSLA **не поддерживает** для македонского: dependency parsing и NER (нет обученных моделей).

Точность лемматизации CLASSLA на оценочных данных: F1 ~ 98.81%.

**Установка:** стандартный `pip install classla` вызывает даунгрейд torch и protobuf из-за устаревших пинов в `setup.py`. Проверено: CLASSLA 2.2.1 работает с актуальными версиями, поэтому ставим через `pip install classla==2.2.1 --no-deps`.

In [ ]:
# установка CLASSLA (выполнить один раз, потом можно закомментировать)
# pip install classla==2.2.1 --no-deps
# pip install obeliks==1.1.6 reldi-tokeniser==1.0.3

import classla
import os

# путь к ресурсам CLASSLA (модели для mk)
resources_dir = os.path.expanduser('~/classla_resources')

# проверяем, скачаны ли модели для македонского
mk_dir = os.path.join(resources_dir, 'mk')
if not os.path.exists(mk_dir):
    # скачиваем модели для македонского языка (tokenize, pos, lemma)
    print("Модели для mk не найдены, скачиваем...")
    classla.download('mk')
else:
    print(f"Модели для mk уже скачаны: {mk_dir}")

# создаем pipeline с тремя процессорами: токенизация, POS-теги, лемматизация
print("\nСоздаем CLASSLA pipeline для mk (tokenize, pos, lemma)...")
nlp = classla.Pipeline('mk', processors='tokenize,pos,lemma')
print("Pipeline создан успешно.")

In [ ]:
def print_analysis(sentence_text, doc):
    """
    выводит результат анализа предложения: слово -> лемма (POS-тег)
    doc -- результат работы pipeline на одном предложении
    """
    print(f"  Предложение: {sentence_text}")
    # проходим по всем предложениям в документе
    for sent in doc.sentences:
        # проходим по всем словам (токенам) в предложении
        for word in sent.words:
            # word.text -- исходная форма, word.lemma -- лемма, word.upos -- POS-тег
            print(f"    {word.text:15s} -> {word.lemma:15s} ({word.upos})")
    print()


# тест 1: базовый пример -- кошка ест большую рыбу в озере
test1 = "Мачката јаде голема риба во езерото."
doc1 = nlp(test1)
print("ТЕСТ 1: базовый пример")
print_analysis(test1, doc1)

# тест 2: существительные с определенными артиклями
# в македонском артикль приклеивается к концу слова: -от, -та, -то
test2 = "Книгата е на масата."  # Книга на столе
doc2 = nlp(test2)
print("ТЕСТ 2: существительные с артиклями")
print_analysis(test2, doc2)

test2b = "Градот е голем."  # Город большой
doc2b = nlp(test2b)
print_analysis(test2b, doc2b)

test2c = "Детето спие."  # Ребенок спит
doc2c = nlp(test2c)
print_analysis(test2c, doc2c)

In [ ]:
# тест 3: глаголы в разных временах
# имперфект: читаше -> чита
test3a = "Тој читаше книга вчера."
doc3a = nlp(test3a)
print("ТЕСТ 3: глаголы в разных временах")
print_analysis(test3a, doc3a)

# перфект с л-причастие: напишала -> напише
test3b = "Таа напишала писмо."
doc3b = nlp(test3b)
print_analysis(test3b, doc3b)

# футур: ќе одам -> оди
test3c = "Јас ќе одам дома."
doc3c = nlp(test3c)
print_analysis(test3c, doc3c)

# тест 4: прилагательные с артиклями
test4a = "Големата куќа е убава."  # Большой дом красивый
doc4a = nlp(test4a)
print("\nТЕСТ 4: прилагательные с артиклями")
print_analysis(test4a, doc4a)

test4b = "Малите деца играат."  # Маленькие дети играют
doc4b = nlp(test4b)
print_analysis(test4b, doc4b)

# тест 5: местоимения
test5 = "Оваа книга е моја."  # Эта книга моя
doc5 = nlp(test5)
print("ТЕСТ 5: местоимения")
print_analysis(test5, doc5)

# тест 6: сложные случаи -- заимствования, неологизмы
test6 = "Компјутерот го процесира барањето."  # Компьютер обрабатывает запрос
doc6 = nlp(test6)
print("ТЕСТ 6: сложные случаи")
print_analysis(test6, doc6)

print(f"Версия CLASSLA: {classla.__version__}")

## 2.2. Морфология македонского языка

Краткий обзор морфологических особенностей, нужных для rule-based лемматизатора.

### Определенные артикли (членови)

Македонский — один из немногих славянских языков с **постпозитивным артиклем** (артикль прикрепляется к концу слова, как в болгарском). Три степени определенности:

| Степень | Муж. род | Жен. род | Средн. род | Мн. число |
|---------|----------|----------|------------|-----------|
| Неопределенно-указательный | -от | -та | -то | -те |
| Близкий | -ов | -ва | -во | -ве |
| Далекий | -он | -на | -но | -не |

Примеры: *мажот* -> *маж*, *куќата* -> *куќа*, *детето* -> *дете*, *книгите* -> *книги*

### Глагольные формы

Три класса спряжения: а-класс (*чита*), е-класс (*земе*), и-класс (*оди*).

- Настоящее время: -ам, -аш, -а, -аме, -ате, -аат (а-класс)
- Имперфект: -в, -ше, -ше, -вме, -вте, -а/-еа
- L-причастие: -л, -ла, -ло, -ле
- Будущее: частица *ќе* + настоящее время

### Существительные

Три рода: мужской (согласная), женский (-а), средний (-о/-е). Множественное число: м.р. -ови/-и, ж.р. -и, ср.р. -а. Лемма — неопределенная форма единственного числа.

### Прилагательные

Согласование по роду, числу, определенности: *голем/голема/големо/големи*. Определенные формы: *големиот/големата/големото/големите*. Сравнительная: *по-* (*поголем*), превосходная: *нај-* (*најголем*). Лемма — м.р. ед.ч. неопределенная форма.

### Местоимения

Полные формы: *јас, ти, тој, таа, тоа, ние, вие, тие*. Краткие (клитические): *ме/ми, го/му, ја/и* и др. Для лемматизации краткие формы приводятся к полной именительной форме.

### Сводная таблица суффиксов для лемматизатора

| Часть речи | Суффиксы для отсечения | Пример |
|------------|----------------------|--------|
| Артикль (все POS) | -от, -та, -то, -ов, -ва, -во, -он, -на, -но, -те, -ве, -не | мажот -> маж |
| Глагол (VERB/AUX) | -л, -ла, -ло, -ле, -ше, -вме, -вте, -м, -ш, -ме, -те, -ат | читаше -> чита |
| Прилагательное (ADJ) | нај-, по-, + артикль + -а, -о, -и | најголемата -> голем |
| Существительное (NOUN) | артикль + -ови, -еви, -ишта, -иња | градови -> град |

## 2.3. Rule-based лемматизатор для македонского языка

**Архитектура:** гибридная схема — сначала lookup-таблица (словарь исключений), потом правила.

- Вход: пара *(словоформа, POS-тег)* — POS-тег берется из CLASSLA
- Выход: лемма (строка)
- Порядок обработки: (1) проверить словарь исключений -> (2) применить правила по POS-тегу -> (3) если ничего не сработало — вернуть форму в нижнем регистре

In [ ]:
"""
Rule-based лемматизатор для македонского языка, часть 1:
словарь исключений, отсечение артиклей, правила для глаголов.
"""


# настраиваем вывод в UTF-8, чтобы кириллица не ломалась на Windows



# словарь хранит пары (форма_в_нижнем_регистре, UPOS-тег) -> лемма
# для некоторых слов ключ без POS — просто строка -> лемма (fallback)
EXCEPTIONS = {}

# глагол "сум" (быть) — все формы без привязки к POS,
# потому что CLASSLA обычно размечает их как AUX, но бывают и VERB
for form in ['е', 'си', 'сме', 'сте', 'бев', 'беше',
             'бил', 'била', 'било', 'биле',
             'беа', 'бевме', 'бевте']:
    # записываем без POS — подходит для любого тега
    EXCEPTIONS[form] = 'сум'

# глагол "има" (иметь)
for form in ['имам', 'имаш', 'имаше', 'имал', 'имала', 'имало', 'имале',
             'имаме', 'имате', 'имаат', 'имав', 'имавме', 'имавте', 'имаа']:
    EXCEPTIONS[form] = 'има'

# глагол "може" (мочь)
for form in ['можам', 'можеш', 'можеше', 'можеме', 'можете', 'можат',
             'можев', 'можевме', 'можевте', 'можеа']:
    EXCEPTIONS[form] = 'може'

# неправильное существительное: луѓе (люди) -> човек (человек)
# добавляем и определённую форму луѓето, потому что лемматизатор
# сначала проверяет исключения, а потом снимает артикль — без этого
# луѓето превратится в луѓе, а не в човек
EXCEPTIONS['луѓе'] = 'човек'
EXCEPTIONS['луѓето'] = 'човек'

# "се" — многозначное слово, привязываем к POS:
# как PRON (возвратная частица "себя") -> себе
EXCEPTIONS[('се', 'PRON')] = 'себе'
# как AUX (3 мн.ч. глагола "быть") -> сум
EXCEPTIONS[('се', 'AUX')] = 'сум'

# краткие местоимения — привязаны к POS=PRON,
# чтобы "не" как частица (PART) не попало в исключения
_SHORT_PRONOUNS = {
    'ме': 'јас',   # меня/мне -> я
    'те': 'ти',    # тебя/тебе -> ты
    'го': 'тој',   # его (м.р.) -> он
    'ја': 'таа',   # её -> она
    'не': 'ние',   # нас -> мы
    'ве': 'вие',   # вас -> вы
    'ги': 'тие',   # их -> они
    'ми': 'јас',   # мне -> я
    'му': 'тој',   # ему -> он
    'и': 'таа',    # ей -> она
    'ни': 'ние',   # нам -> мы
    'ви': 'вие',   # вам -> вы
    'им': 'тие',   # им -> они
    'нам': 'ние',  # нам (полная форма) -> мы
    'вам': 'вие',  # вам (полная форма) -> вы
    'нас': 'ние',  # нас (полная форма) -> мы
    'вас': 'вие',  # вас (полная форма) -> вы
    'нив': 'тие',  # них (после предлога: во нив) -> они
    'него': 'тој', # него (после предлога: за него) -> он
    'неа': 'таа',  # неё (после предлога: за неа) -> она
    'нејзе': 'таа', # ей (после предлога: кон нејзе) -> она
}

# записываем краткие местоимения с ключом (форма, 'PRON')
for form, lemma in _SHORT_PRONOUNS.items():
    EXCEPTIONS[(form, 'PRON')] = lemma

# личные местоимения 3 лица: они же леммы, но без исключения
# strip_article их портит (таа -> та, тоа -> то), поэтому явно фиксируем
EXCEPTIONS[('таа', 'PRON')] = 'таа'
EXCEPTIONS[('тој', 'PRON')] = 'тој'
EXCEPTIONS[('тоа', 'PRON')] = 'тоа'

# указательные местоимения — привязаны к POS=DET или PRON
_DEMONSTRATIVES = {
    'оваа': 'овој',  # эта -> этот
    'ова': 'овој',    # это -> этот
    'овие': 'овој',   # эти -> этот
    'онаа': 'оној',   # та (ж.р.) -> тот
    'она': 'оној',    # то (ср.р.) -> тот
    'оние': 'оној',   # те -> тот
    'тоа': 'тој',     # то (ср.р.) -> тој
    'тие': 'тој',     # те (мн.ч.) -> тој
}

# указательные работают и как DET, и как PRON — добавляем оба варианта
for form, lemma in _DEMONSTRATIVES.items():
    EXCEPTIONS[(form, 'DET')] = lemma
    EXCEPTIONS[(form, 'PRON')] = lemma


def lookup_exception(form, upos):
    """
    Ищем форму в словаре исключений.
    Сначала пробуем точный ключ (форма, UPOS),
    потом fallback — просто форму без POS.
    Возвращаем лемму или None, если не нашли.
    """
    # приводим форму к нижнему регистру
    form_lower = form.lower()

    # сначала ищем с привязкой к POS-тегу
    result = EXCEPTIONS.get((form_lower, upos))
    if result is not None:
        return result

    # если не нашли — ищем без POS (fallback)
    result = EXCEPTIONS.get(form_lower)
    if result is not None:
        return result

    # ничего не нашли
    return None



# суффиксы-артикли: только двухбуквенные
# трёхбуквенные из плана (вроде -ите, -ата, -ото) на самом деле состоят из
# гласной основы + двухбуквенного артикля: книги+те=книгите, мачка+та=мачката
# если отрезать -ите от "книгите", получится "книг" вместо "книги" — слишком много
# поэтому отрезаем только сам артикль (2 буквы), а гласную основы оставляем
_ARTICLE_SUFFIXES = [
    'от', 'та', 'то',  # медијален: маж+от, куќа+та, дете+то
    'ов', 'ва', 'во',  # блиски: маж+ов, куќа+ва, дете+во
    'он', 'на', 'но',  # далечен: маж+он, куќа+на, дете+но
    'те', 'ве', 'не',  # мн.ч.: книги+те, книги+ве, книги+не
]

# POS-теги, для которых имеет смысл отсекать артикль
_ARTICLE_POS_SET = {'NOUN', 'ADJ', 'NUM', 'DET', 'PRON'}

# минимальная длина основы, которая должна остаться после отсечения
_MIN_STEM_LENGTH = 2


def strip_article(form, upos):
    """
    Отсекаем определённый артикль от словоформы.
    Работает только для существительных, прилагательных, числительных,
    детерминаторов и местоимений.
    Возвращаем форму без артикля или исходную форму, если ничего не подошло.
    """
    # проверяем, что POS-тег подходит для отсечения артикля
    if upos not in _ARTICLE_POS_SET:
        return form

    # перебираем суффиксы от длинных к коротким
    for suffix in _ARTICLE_SUFFIXES:
        # проверяем, что форма заканчивается на этот суффикс
        if form.endswith(suffix):
            # отрезаем суффикс
            stem = form[:-len(suffix)]
            # проверяем, что основа не слишком короткая
            if len(stem) >= _MIN_STEM_LENGTH:
                return stem

    # ни один суффикс не подошёл — возвращаем как есть
    return form



# суффиксы для L-причастия (прошедшее время):
# читала, читало, читале, читал
_L_PARTICIPLE_SUFFIXES = ['ла', 'ло', 'ле', 'л']

# суффиксы настоящего времени — только личные окончания без тематической гласной
# тематическая гласная (а/е/и) принадлежит основе, а не окончанию:
# читам = чита+м, читаш = чита+ш, земеш = земе+ш, одиш = оди+ш
# если отрезать -ам от "читам", получится "чит" вместо "чита"
# порядок: от длинных к коротким
_PRESENT_SUFFIXES = [
    'ме',   # 1 мн.: читаме->чита, земеме->земе, одиме->оди
    'те',   # 2 мн.: читате->чита, земете->земе, одите->оди
    'ат',   # 3 мн.: читаат->чита, земат->зем, одат->од (е/и-класс неточно)
    'ш',    # 2 ед.: читаш->чита, земеш->земе, одиш->оди
    'м',    # 1 ед.: читам->чита
]

# суффиксы имперфекта и аориста:
# читавме, читавте, читаше, читав
_IMPERFECT_AORIST_SUFFIXES = ['вме', 'вте', 'ше', 'в']

# суффиксы повелительного наклонения:
# читајте, читај
_IMPERATIVE_SUFFIXES = ['јте', 'ј']

# минимальная длина основы для глаголов
_VERB_MIN_STEM = 2


def lemmatize_verb(form):
    """
    Пробуем лемматизировать глагольную форму, отсекая суффиксы.
    Форма уже должна быть в нижнем регистре.
    Возвращаем предполагаемую лемму (основу) или исходную форму.
    """
    # пробуем группы суффиксов по порядку:
    # L-причастие, повелительное (до настоящего времени, потому что -јте длиннее -те),
    # имперфект/аорист, настоящее время (в конце, потому что суффиксы короткие)
    all_suffix_groups = [
        _L_PARTICIPLE_SUFFIXES,
        _IMPERATIVE_SUFFIXES,
        _IMPERFECT_AORIST_SUFFIXES,
        _PRESENT_SUFFIXES,
    ]

    for group in all_suffix_groups:
        for suffix in group:
            # проверяем, заканчивается ли форма на этот суффикс
            if form.endswith(suffix):
                # отрезаем суффикс
                stem = form[:-len(suffix)]
                # проверяем минимальную длину основы
                if len(stem) >= _VERB_MIN_STEM:
                    return stem

    # ни одно правило не сработало — возвращаем как есть
    return form




In [ ]:
# правила лемматизации для прилагательных, существительных
# и главная функция rule_based_lemmatize
# (часть 2 из 2 — зависит от step4_1_4_rules_part1.py)

import re

# настраиваем кодировку stdout для Windows, чтобы кириллица не ломалась

# добавляем папку temp в путь поиска модулей

# импортируем всё, что нужно, из первой части (словарь исключений, снятие артикля, глаголы)


def lemmatize_adj(form):
    """
    Лемматизация прилагательного: снимаем степень сравнения,
    артикль, родовое окончание. Возвращаем м.р. ед.ч. неопр. форму.
    """
    # работаем с копией, чтобы не портить исходную форму
    stem = form

    # шаг 1: снятие приставок степеней сравнения
    # сначала проверяем превосходную степень (нај-), потом сравнительную (по-)
    # порядок важен: "најпо..." — сначала снимаем "нај", потом "по"
    if stem.startswith('нај') and len(stem) > len('нај') + 2:
        # убираем приставку превосходной степени
        stem = stem[len('нај'):]

    if stem.startswith('по') and len(stem) > len('по') + 2:
        # убираем приставку сравнительной степени
        stem = stem[len('по'):]

    # шаг 2: снятие артикля через общую функцию из part1
    stem = strip_article(stem, 'ADJ')

    # шаг 3: снятие родового окончания (ж.р. -а, ср.р. -о, мн.ч. -и)
    # после снятия артикля у м.р. определённой формы может остаться -и
    # (например: големиот -> големи -> голем)
    if len(stem) > 2 and stem.endswith('а'):
        # женский род: голема -> голем
        stem = stem[:-1]
    elif len(stem) > 2 and stem.endswith('о'):
        # средний род: големо -> голем
        stem = stem[:-1]
    elif len(stem) > 2 and stem.endswith('и'):
        # множественное число или остаток от м.р. определённой формы
        stem = stem[:-1]

    return stem


def lemmatize_noun(form):
    """
    Лемматизация существительного: снимаем артикль, потом суффикс мн. числа.
    Возвращаем ед.ч. неопределённую форму.
    """
    # шаг 1: снятие артикля через общую функцию из part1
    stem = strip_article(form, 'NOUN')

    # шаг 2: пробуем снять суффиксы множественного числа
    # только однозначные длинные суффиксы: -ишта, -ови, -еви, -иња
    # короткие -и и -а не трогаем: они совпадают с окончанием женского рода (-а)
    # и множественного числа (-и), что даёт ложные срабатывания
    # (мачка -> мачк, вода -> вод) — лучше оставить форму как есть
    plural_suffixes = ['ишта', 'ови', 'еви', 'иња']

    for suffix in plural_suffixes:
        # проверяем, что слово заканчивается на этот суффикс
        if stem.endswith(suffix):
            # вычисляем, какая основа останется после отрезания
            candidate = stem[:-len(suffix)]
            # основа должна быть хотя бы 2 символа
            if len(candidate) >= 2:
                # снимаем суффикс и возвращаем основу
                return candidate

    # если ни один суффикс не подошёл, возвращаем то, что есть (уже без артикля)
    return stem


def rule_based_lemmatize(form, upos):
    """
    Главная функция rule-based лемматизатора.
    Принимает словоформу и POS-тег (UPOS), возвращает лемму.
    """
    # крайний случай: пунктуация — возвращаем как есть, без изменений
    if upos == 'PUNCT':
        return form

    # приводим форму к нижнему регистру для единообразия
    form_lower = form.lower()

    # крайний случай: числа (цифры, может с точкой или запятой) — не трогаем
    if re.fullmatch(r'[\d.,]+', form_lower):
        return form_lower

    # крайний случай: слова на латинице — возвращаем в нижнем регистре
    if re.fullmatch(r'[a-zA-Z]+', form):
        return form_lower

    # проверяем словарь исключений — если слово там есть, берём готовую лемму
    # важно проверить ДО фильтра по длине, потому что "е" (AUX) — одна буква, но это форма "сум"
    exception = lookup_exception(form_lower, upos)
    if exception is not None:
        return exception

    # крайний случай: однобуквенные слова — нечего лемматизировать
    if len(form_lower) == 1:
        return form_lower

    # основная логика: выбираем правила в зависимости от части речи
    if upos in ('VERB', 'AUX'):
        # глаголы и вспомогательные глаголы — правила из part1
        return lemmatize_verb(form_lower)

    elif upos == 'ADJ':
        # прилагательные — наша функция из этого файла
        return lemmatize_adj(form_lower)

    elif upos == 'NOUN':
        # существительные — наша функция из этого файла
        return lemmatize_noun(form_lower)

    elif upos in ('NUM', 'DET'):
        # числительные и определители: снимаем артикль, потом родовые окончания
        stem = strip_article(form_lower, upos)
        # пробуем снять родовые окончания -и, -а, -о
        if len(stem) > 2 and stem.endswith('и'):
            stem = stem[:-1]
        elif len(stem) > 2 and stem.endswith('а'):
            stem = stem[:-1]
        elif len(stem) > 2 and stem.endswith('о'):
            stem = stem[:-1]
        return stem

    elif upos == 'PRON':
        # местоимения: большинство покрыты исключениями (проверили выше)
        # для остальных пробуем снять артикль и родовые окончания
        stem = strip_article(form_lower, upos)
        if len(stem) > 2 and stem.endswith('а'):
            stem = stem[:-1]
        elif len(stem) > 2 and stem.endswith('о'):
            stem = stem[:-1]
        elif len(stem) > 2 and stem.endswith('и'):
            stem = stem[:-1]
        return stem

    elif upos == 'ADV':
        # наречия обычно не меняются — возвращаем как есть
        return form_lower

    else:
        # все остальные части речи (CCONJ, SCONJ, ADP, PART, INTJ, X...)
        return form_lower

In [ ]:
# проверяем rule-based лемматизатор на нескольких примерах
test_cases = [
    ('Мачката', 'NOUN', 'мачка'),
    ('голема', 'ADJ', 'голем'),
    ('читаше', 'VERB', 'чита'),
    ('.', 'PUNCT', '.'),
    ('го', 'PRON', 'тој'),
    ('е', 'AUX', 'сум'),
    ('градови', 'NOUN', 'град'),
    ('најголемата', 'ADJ', 'голем'),
]

print("Проверка rule-based лемматизатора:")
print(f"  {'форма':<18s} {'POS':<6s} {'результат':<15s} {'ожидали':<15s} {'OK?'}")
for form, upos, expected in test_cases:
    result = rule_based_lemmatize(form, upos)
    ok = 'OK' if result == expected else 'FAIL'
    print(f"  {form:<18s} {upos:<6s} {result:<15s} {expected:<15s} {ok}")

### Обертка для обработки текста rule-based лемматизатором

Функция `process_text_rulebased` имеет ту же сигнатуру, что и `process_text_classla`: принимает текст и CLASSLA pipeline, возвращает *(sentences, log_info)*. Внутри она использует CLASSLA для токенизации и POS-тегирования, а лемматизацию заменяет на результат наших правил.

In [ ]:
import time


def process_text_rulebased(text, nlp_pipeline):
    """
    обрабатывает текст: токенизация и POS через CLASSLA, лемматизация через наши правила

    сигнатура полностью совпадает с process_text_classla(text, nlp_pipeline):
      text         -- строка с текстом
      nlp_pipeline -- загруженный CLASSLA pipeline (нужен для tokenize + pos)

    возвращает тот же формат:
      sentences -- список предложений, каждое -- список словарей
                   {"form", "lemma", "upos", "xpos"}
      log_info  -- словарь {"n_sentences", "n_tokens", "n_chunks", "errors", "elapsed_sec"}
    """
    # засекаем время
    start_time = time.time()

    # шаг 1: прогоняем текст через CLASSLA для получения токенов и POS-тегов
    sentences, classla_log = process_text_classla(text, nlp_pipeline)

    # шаг 2: заменяем лемму CLASSLA на нашу rule-based лемму в каждом токене
    for sentence in sentences:
        for token in sentence:
            form = token['form']
            upos = token['upos']
            # заменяем лемму на результат нашего rule-based лемматизатора
            token['lemma'] = rule_based_lemmatize(form, upos)

    # считаем итоговое время
    elapsed = time.time() - start_time

    log_info = {
        'n_sentences': classla_log['n_sentences'],
        'n_tokens': classla_log['n_tokens'],
        'n_chunks': classla_log['n_chunks'],
        'errors': classla_log['errors'],
        'elapsed_sec': round(elapsed, 2)
    }

    return sentences, log_info

## 2.4. SQLite-база для хранения NLP-результатов

Все результаты NLP-обработки хранятся в единой базе `datasets/nlp_data.db`. Зачем SQLite вместо CSV:

1. **Промежуточное сохранение** — после каждого текста результат уже в базе, при перезапуске продолжаем с прерванного места
2. **Скорость** — миллионы строк-токенов в CSV нечитаемы и медленно грузятся, SQLite с индексами — мгновенный доступ
3. **Единая точка хранения** — для обоих лемматизаторов, gold standard и логов обработки
4. **Совместимость с pandas** — `pd.read_sql("SELECT * FROM classla_tokens WHERE text_id = ?", conn, params=[42])`

**SQLite** — встроенная в Python база данных, не нужен сервер, хранится в одном файле.

In [ ]:

import sqlite3
import os
import pandas as pd

# принудительно ставим UTF-8 для корректного вывода кириллицы в Windows-консоли


class NlpDatabase:
    """
    класс-обертка над SQLite-базой для хранения NLP-результатов
    поддерживает два метода (classla, rulebased), gold standard и лог обработки
    работает как context manager (with NlpDatabase(...) as db:)
    """

    def __init__(self, db_path):
        # запоминаем путь к файлу базы
        self.db_path = db_path
        # создаем папку для базы, если она еще не существует
        os.makedirs(os.path.dirname(db_path), exist_ok=True)
        # открываем соединение с SQLite-базой (создаст файл, если его нет)
        self.conn = sqlite3.connect(db_path)
        # включаем режим словарей — строки будут возвращаться как sqlite3.Row
        self.conn.row_factory = sqlite3.Row
        # создаем таблицы и индексы, если их еще нет
        self._create_tables()

    def _create_tables(self):
        # получаем курсор для выполнения SQL-запросов
        cur = self.conn.cursor()

        # таблица для токенов, полученных через CLASSLA
        # text_id — номер текста из корпуса
        # sentence_id — номер предложения внутри текста
        # token_id — номер токена внутри предложения
        # form — словоформа как в тексте, lemma — лемма, upos — универсальный POS-тег, xpos — расширенный POS-тег
        cur.execute("""
            CREATE TABLE IF NOT EXISTS classla_tokens (
                text_id     INTEGER,
                sentence_id INTEGER,
                token_id    INTEGER,
                form        TEXT,
                lemma       TEXT,
                upos        TEXT,
                xpos        TEXT,
                PRIMARY KEY (text_id, sentence_id, token_id)
            )
        """)

        # таблица для токенов, полученных rule-based лемматизатором (тот же формат)
        cur.execute("""
            CREATE TABLE IF NOT EXISTS rulebased_tokens (
                text_id     INTEGER,
                sentence_id INTEGER,
                token_id    INTEGER,
                form        TEXT,
                lemma       TEXT,
                upos        TEXT,
                xpos        TEXT,
                PRIMARY KEY (text_id, sentence_id, token_id)
            )
        """)

        # таблица для gold standard — вручную проверенных токенов
        # source — откуда взят эталон (например, "manual" или "wikiann")
        # sentence_id и token_id — позиция в gold-корпусе
        # lemma_gold — эталонная лемма для сравнения с CLASSLA и rule-based
        cur.execute("""
            CREATE TABLE IF NOT EXISTS gold_standard (
                source      TEXT,
                sentence_id INTEGER,
                token_id    INTEGER,
                form        TEXT,
                lemma_gold  TEXT,
                upos        TEXT
            )
        """)

        # таблица логов обработки — один текст может быть обработан разными методами
        # поэтому PRIMARY KEY составной: (text_id, method)
        # status — "ok" или "error"
        # n_sentences, n_tokens — количество предложений и токенов
        # elapsed_sec — время обработки в секундах
        # error — текст ошибки (NULL если все прошло нормально)
        cur.execute("""
            CREATE TABLE IF NOT EXISTS processing_log (
                text_id     INTEGER,
                method      TEXT,
                status      TEXT,
                n_sentences INTEGER,
                n_tokens    INTEGER,
                elapsed_sec REAL,
                error       TEXT,
                PRIMARY KEY (text_id, method)
            )
        """)

        # индексы для быстрого поиска токенов по text_id и sentence_id
        # IF NOT EXISTS — чтобы не падать при повторном запуске
        cur.execute("""
            CREATE INDEX IF NOT EXISTS idx_classla_text
            ON classla_tokens (text_id)
        """)
        cur.execute("""
            CREATE INDEX IF NOT EXISTS idx_classla_sentence
            ON classla_tokens (text_id, sentence_id)
        """)
        cur.execute("""
            CREATE INDEX IF NOT EXISTS idx_rulebased_text
            ON rulebased_tokens (text_id)
        """)
        cur.execute("""
            CREATE INDEX IF NOT EXISTS idx_rulebased_sentence
            ON rulebased_tokens (text_id, sentence_id)
        """)

        # фиксируем изменения в базе
        self.conn.commit()

    def _table_for_method(self, method):
        # выбираем таблицу по названию метода
        if method == 'classla':
            return 'classla_tokens'
        elif method == 'rulebased':
            return 'rulebased_tokens'
        else:
            # если передали неизвестный метод — сразу сообщаем об ошибке
            raise ValueError(f"неизвестный метод: {method!r}, допустимо: 'classla' или 'rulebased'")

    def save_tokens(self, text_id, tokens, method):
        """
        сохраняет список токенов в нужную таблицу (classla_tokens или rulebased_tokens)
        tokens — список словарей вида:
            [{'sentence_id': 0, 'token_id': 1, 'form': 'Мачка', 'lemma': 'мачка', 'upos': 'NOUN', 'xpos': 'Ncfsn'}, ...]
        если токены для этого text_id уже есть — перезаписываем (DELETE + INSERT)
        автоматически коммитит в конце для надежности при сбоях
        """
        # определяем, в какую таблицу писать
        table = self._table_for_method(method)
        # получаем курсор
        cur = self.conn.cursor()
        # сначала удаляем старые токены для этого текста (если были)
        cur.execute(f"DELETE FROM {table} WHERE text_id = ?", (text_id,))
        # вставляем новые токены пачкой через executemany
        cur.executemany(
            f"INSERT INTO {table} (text_id, sentence_id, token_id, form, lemma, upos, xpos) "
            f"VALUES (?, ?, ?, ?, ?, ?, ?)",
            [
                (
                    text_id,
                    t['sentence_id'],
                    t['token_id'],
                    t['form'],
                    t['lemma'],
                    t['upos'],
                    t.get('xpos')  # xpos может отсутствовать — тогда будет NULL
                )
                for t in tokens
            ]
        )
        # коммитим сразу — если скрипт упадет, уже сохраненные тексты не потеряются
        self.conn.commit()

    def get_tokens(self, text_id, method):
        """
        возвращает токены для конкретного текста в виде списка словарей
        каждый словарь содержит: sentence_id, token_id, form, lemma, upos, xpos
        """
        # определяем таблицу
        table = self._table_for_method(method)
        # выбираем все токены для этого text_id, сортируем по предложению и позиции
        cur = self.conn.cursor()
        cur.execute(
            f"SELECT sentence_id, token_id, form, lemma, upos, xpos "
            f"FROM {table} "
            f"WHERE text_id = ? "
            f"ORDER BY sentence_id, token_id",
            (text_id,)
        )
        # превращаем sqlite3.Row в обычные словари
        rows = cur.fetchall()
        return [dict(row) for row in rows]

    def get_processed_ids(self, method):
        """
        возвращает set из text_id, которые уже успешно обработаны этим методом
        нужно для resume — чтобы не обрабатывать тексты заново после перезапуска
        """
        cur = self.conn.cursor()
        # ищем в логе записи со статусом "ok" для нужного метода
        cur.execute(
            "SELECT text_id FROM processing_log WHERE method = ? AND status = 'ok'",
            (method,)
        )
        # собираем text_id в set для быстрой проверки через `in`
        return {row['text_id'] for row in cur.fetchall()}

    def save_log(self, text_id, method, status, n_sentences, n_tokens, elapsed_sec, error=None):
        """
        записывает или обновляет лог обработки текста
        INSERT OR REPLACE — если запись с таким (text_id, method) уже есть, перезапишет
        """
        cur = self.conn.cursor()
        cur.execute(
            "INSERT OR REPLACE INTO processing_log "
            "(text_id, method, status, n_sentences, n_tokens, elapsed_sec, error) "
            "VALUES (?, ?, ?, ?, ?, ?, ?)",
            (text_id, method, status, n_sentences, n_tokens, elapsed_sec, error)
        )
        # коммитим лог сразу — чтобы при сбое знать, какие тексты уже обработаны
        self.conn.commit()

    def to_dataframe(self, method, text_id=None):
        """
        возвращает pandas DataFrame с токенами из нужной таблицы
        если text_id задан — только для конкретного текста, иначе — все тексты
        """
        # определяем таблицу
        table = self._table_for_method(method)
        # формируем SQL-запрос
        if text_id is not None:
            # запрос с фильтром по text_id
            query = f"SELECT * FROM {table} WHERE text_id = ? ORDER BY text_id, sentence_id, token_id"
            # pandas.read_sql_query принимает параметры через params
            df = pd.read_sql_query(query, self.conn, params=(text_id,))
        else:
            # все токены из таблицы
            query = f"SELECT * FROM {table} ORDER BY text_id, sentence_id, token_id"
            df = pd.read_sql_query(query, self.conn)
        return df

    def save_gold_standard(self, records):
        """
        сохраняет записи gold standard пачкой
        records — список словарей:
            [{'source': 'manual', 'sentence_id': 0, 'token_id': 1, 'form': '...', 'lemma_gold': '...', 'upos': '...'}, ...]
        """
        cur = self.conn.cursor()
        cur.executemany(
            "INSERT INTO gold_standard (source, sentence_id, token_id, form, lemma_gold, upos) "
            "VALUES (?, ?, ?, ?, ?, ?)",
            [
                (
                    r['source'],
                    r['sentence_id'],
                    r['token_id'],
                    r['form'],
                    r['lemma_gold'],
                    r['upos']
                )
                for r in records
            ]
        )
        # коммитим
        self.conn.commit()

    def close(self):
        # закрываем соединение с базой
        if self.conn:
            self.conn.close()
            self.conn = None

    def __enter__(self):
        # поддержка context manager — возвращаем self при входе в with-блок
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        # при выходе из with-блока закрываем соединение
        self.close()
        # не подавляем исключения — пусть всплывают наверх
        return False

## 2.5. Улучшенная функция обработки текстов через CLASSLA

Существующий ноутбук `artifacts/тексты_и_лемматизатор_македонский.ipynb` имел серьезные проблемы:

1. **Разрезание слов на границах чанков** — текст резался ровно по 5000-му символу, слово могло попасть в два чанка (*маке|донски*)
2. **Потеря границ предложений** — все токены склеивались в плоский поток через пробел
3. **Нет формата CoNLL** — для построения графов нужен структурированный формат, а не плоский список
4. **Нет обработки ошибок** — если CLASSLA падала на чанке, весь процесс останавливался
5. **Нет продолжения с прерванного места** — 120 из 128 файлов обработаны, процесс прервался

**Наша улучшенная функция:**
- Разбивает текст на чанки по границам предложений (а не по символам)
- Сохраняет границы предложений в выходных данных
- Оборачивает вызов CLASSLA в try/except
- Возвращает структурированный результат: список предложений, каждое — список словарей

In [ ]:
import re
import logging
import time

# принудительно ставим UTF-8, чтобы кириллица не ломалась в Windows-консоли

# настраиваем логирование: будем писать и в консоль, и в файл
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
# создаем логгер для нашего модуля
logger = logging.getLogger('classla_processing')


def split_into_chunks(text, max_chunk_size=5000):
    """
    Разбивает текст на чанки по границам предложений.

    CLASSLA сам умеет сегментировать предложения внутри чанка,
    но не справляется с очень длинными текстами целиком (память, скорость).
    Поэтому мы сначала делаем pre-split на куски разумного размера,
    а CLASSLA потом разбирает каждый кусок внутри.

    Логика:
    1. Разбиваем текст по концам предложений (. ! ? + пробел или перенос)
    2. Собираем фрагменты в чанки, пока длина не превышает max_chunk_size
    3. Если один фрагмент длиннее max_chunk_size — дополнительно режем по ближайшему пробелу
    """
    # если текст короче лимита, возвращаем как есть
    if len(text) <= max_chunk_size:
        return [text]

    # разбиваем текст по концам предложений: точка/!/? + пробел или перенос строки
    # re.split с группой сохраняет разделитель в списке
    fragments = re.split(r'(?<=[.!?])(?=\s)', text)

    chunks = []
    # текущий чанк, который мы собираем
    current_chunk = ""

    for fragment in fragments:
        # если фрагмент сам по себе длиннее лимита — режем по пробелам
        if len(fragment) > max_chunk_size:
            # сначала сбрасываем текущий накопленный чанк, если он непустой
            if current_chunk.strip():
                chunks.append(current_chunk)
                current_chunk = ""
            # режем длинный фрагмент по пробелам
            sub_chunks = _split_long_fragment(fragment, max_chunk_size)
            # добавляем все подчанки, кроме последнего (его оставим для склейки с продолжением)
            for sc in sub_chunks[:-1]:
                chunks.append(sc)
            # последний подчанок начинаем склеивать с дальнейшим текстом
            current_chunk = sub_chunks[-1] if sub_chunks else ""
            continue

        # проверяем, влезет ли фрагмент в текущий чанк
        if len(current_chunk) + len(fragment) <= max_chunk_size:
            # влезает — добавляем к текущему чанку
            current_chunk += fragment
        else:
            # не влезает — сбрасываем текущий чанк и начинаем новый
            if current_chunk.strip():
                chunks.append(current_chunk)
            current_chunk = fragment

    # не забываем про остаток
    if current_chunk.strip():
        chunks.append(current_chunk)

    return chunks


def _split_long_fragment(fragment, max_size):
    """
    Режет слишком длинный фрагмент (> max_size символов) по ближайшему пробелу.
    Не разрезает слова пополам — всегда ищет пробел.
    """
    pieces = []
    # пока фрагмент длиннее лимита, отрезаем куски
    while len(fragment) > max_size:
        # ищем последний пробел в пределах max_size символов
        cut_pos = fragment.rfind(' ', 0, max_size)
        if cut_pos == -1:
            # если пробела нет (одно гигантское слово) — режем по лимиту
            cut_pos = max_size
        # отрезаем кусок до пробела
        pieces.append(fragment[:cut_pos])
        # оставшуюся часть продолжаем разрезать (пропускаем пробел на границе)
        fragment = fragment[cut_pos:].lstrip()

    # остаток — тоже добавляем
    if fragment.strip():
        pieces.append(fragment)

    return pieces


def process_text_classla(text, nlp_pipeline):
    """
    Шаг 2.3: Улучшенная функция обработки текста через CLASSLA.

    Принимает:
      text         — строка с полным текстом
      nlp_pipeline — загруженный CLASSLA pipeline (classla.Pipeline)

    Возвращает:
      result   — список предложений; каждое предложение — список словарей
                 {"form": ..., "lemma": ..., "upos": ..., "xpos": ...}
      log_info — словарь с логом обработки:
                 {"n_sentences": ..., "n_tokens": ..., "n_chunks": ...,
                  "errors": [...], "elapsed_sec": ...}

    Формат результата:
    [
        [  # предложение 0
            {"form": "Мачката", "lemma": "мачка", "upos": "NOUN", "xpos": "Ncfsd"},
            {"form": "јаде", "lemma": "јаде", "upos": "VERB", "xpos": "Vmr3s"},
            ...
        ],
        [  # предложение 1
            ...
        ],
    ]
    """
    # засекаем время начала обработки
    start_time = time.time()

    # сюда будем складывать все предложения из всех чанков
    all_sentences = []
    # сюда пишем ошибки, если какой-то чанк не обработался
    errors = []

    # разбиваем текст на чанки по границам предложений
    chunks = split_into_chunks(text, max_chunk_size=5000)
    n_chunks = len(chunks)
    logger.info(f"текст разбит на {n_chunks} чанков")

    for i, chunk in enumerate(chunks):
        # пропускаем пустые чанки
        if not chunk.strip():
            continue

        try:
            # прогоняем чанк через CLASSLA pipeline
            doc = nlp_pipeline(chunk)

            # проходим по всем предложениям, которые CLASSLA нашел в чанке
            for sentence in doc.sentences:
                # собираем токены одного предложения
                sentence_tokens = []
                for word in sentence.words:
                    # записываем все четыре поля: форму, лемму, универсальный POS, расширенный POS
                    token_info = {
                        "form": word.text,
                        "lemma": word.lemma,
                        "upos": word.upos,
                        "xpos": word.xpos if word.xpos else ""
                    }
                    sentence_tokens.append(token_info)

                # добавляем предложение в общий список (только непустые)
                if sentence_tokens:
                    all_sentences.append(sentence_tokens)

        except Exception as e:
            # ловим любую ошибку CLASSLA и логируем, но не останавливаемся
            error_msg = f"чанк {i+1}/{n_chunks}: {type(e).__name__}: {e}"
            logger.warning(f"ошибка при обработке: {error_msg}")
            errors.append(error_msg)

    # считаем итоговую статистику
    elapsed = time.time() - start_time
    n_sentences = len(all_sentences)
    # общее число токенов — сумма длин всех предложений
    n_tokens = sum(len(s) for s in all_sentences)

    # собираем лог обработки
    log_info = {
        "n_sentences": n_sentences,
        "n_tokens": n_tokens,
        "n_chunks": n_chunks,
        "errors": errors,
        "elapsed_sec": round(elapsed, 2)
    }

    logger.info(
        f"готово: {n_sentences} предложений, {n_tokens} токенов, "
        f"{len(errors)} ошибок, {elapsed:.2f} сек"
    )

    return all_sentences, log_info


def print_sentences(sentences, max_sentences=None):
    """
    Выводит результат в читаемом виде: форма -> лемма (POS).
    max_sentences — если задано, выведет только первые N предложений.
    """
    # определяем, сколько предложений выводить
    limit = max_sentences if max_sentences else len(sentences)
    for i, sentence in enumerate(sentences[:limit]):
        print(f"\n  предложение {i}:")
        for token in sentence:
            # формат: форма -> лемма (UPOS)
            print(f"    {token['form']:20s} -> {token['lemma']:20s} ({token['upos']})")

In [ ]:
import os

# принудительно ставим UTF-8, чтобы кириллица не ломалась в Windows-консоли


def to_conll_string(sentences):
    """
    конвертирует список предложений из формата CLASSLA в строку CoNLL

    вход: список предложений, каждое предложение — список словарей
          с полями form, lemma, upos, xpos
          (такой формат возвращает process_text_classla)

    выход: строка в формате CoNLL, совместимая с sent_class.parse_conll()
           формат каждой строки: ID\tFORM\tLEMMA\tUPOS\tXPOS\t_\tHEAD\tDEPREL\t_\t_

    поля HEAD и DEPREL заполнены заглушками (0 и _), потому что
    CLASSLA для македонского не поддерживает dependency parsing.
    parse_conll() из sent_class.py их корректно прочитает —
    реальный dependency parsing будет добавлен позже через spaCy (шаг 9)
    """
    # собираем все строки в список, потом склеим через перевод строки
    lines = []

    # проходим по каждому предложению
    for sent in sentences:
        # склеиваем текст предложения из форм токенов для комментария # text = ...
        sentence_text = " ".join(token["form"] for token in sent)

        # добавляем комментарий с текстом предложения (parse_conll его читает)
        lines.append(f"# text = {sentence_text}")

        # проходим по токенам предложения, нумеруя с 1
        for token_id, token in enumerate(sent, start=1):
            # достаем поля из словаря токена
            form = token["form"]
            lemma = token["lemma"]
            upos = token["upos"]
            # xpos может отсутствовать — тогда ставим прочерк
            xpos = token.get("xpos", "_")

            # собираем строку в формате CoNLL: 10 полей через табуляцию
            # поля 6 (feats), 9 (deps), 10 (misc) — прочерки
            # поле 7 (HEAD) = 0 (заглушка: все токены указывают на root)
            # поле 8 (DEPREL) = _ (заглушка: нет dependency parsing)
            conll_line = f"{token_id}\t{form}\t{lemma}\t{upos}\t{xpos}\t_\t0\t_\t_\t_"
            lines.append(conll_line)

        # пустая строка после каждого предложения (разделитель в CoNLL)
        lines.append("")

    # склеиваем все строки через перевод строки
    return "\n".join(lines)


def to_conll_file(sentences, filepath):
    """
    записывает результат конвертации в CoNLL-файл

    вход: sentences — список предложений (тот же формат, что и для to_conll_string)
          filepath — путь к файлу для записи

    выход: файл на диске в формате CoNLL (UTF-8)
    """
    # конвертируем предложения в строку CoNLL
    conll_text = to_conll_string(sentences)

    # записываем в файл с кодировкой UTF-8
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(conll_text)

    # сообщаем, куда записали и сколько предложений
    print(f"CoNLL-файл записан: {filepath}")
    print(f"  предложений: {len(sentences)}")

    # считаем общее количество токенов во всех предложениях
    total_tokens = sum(len(sent) for sent in sentences)
    print(f"  токенов: {total_tokens}")


def sentences_to_flat_tokens(sentences, text_id=0):
    """
    конвертирует список предложений в плоский список токенов
    для записи в SQLite через NlpDatabase.save_tokens()

    вход: sentences — список предложений (формат process_text_classla)
          text_id — идентификатор текста (по умолчанию 0)

    выход: список словарей, каждый словарь — один токен с полями:
           text_id, sentence_id, token_id, form, lemma, upos, xpos
    """
    # собираем плоский список токенов
    flat_tokens = []

    # проходим по предложениям, нумеруя с 0
    for sentence_id, sent in enumerate(sentences):
        # проходим по токенам внутри предложения, нумеруя с 1
        for token_id, token in enumerate(sent, start=1):
            # создаем словарь с полными координатами токена в корпусе
            flat_token = {
                "text_id": text_id,
                "sentence_id": sentence_id,
                "token_id": token_id,
                "form": token["form"],
                "lemma": token["lemma"],
                "upos": token["upos"],
                "xpos": token.get("xpos", "_"),
            }
            flat_tokens.append(flat_token)

    return flat_tokens

## 2.6. Обработка корпуса

### CLASSLA: обработка 98 текстов

Обработка всего корпуса через CLASSLA заняла ~50 минут на CPU. Результаты сохранены в таблице `classla_tokens` в `datasets/nlp_data.db`.

Код ниже показывает, как обработка была запущена. Результаты уже в базе, поэтому повторный запуск не нужен.

In [ ]:
# флаг для контроля тяжелых вычислений
# поставить True, чтобы запустить обработку заново (займет ~50 мин)
RUN_HEAVY = False

import pandas as pd

if RUN_HEAVY:
    # загружаем корпус текстов
    corpus_path = r'datasets/raw/corpus_full.csv'
    corpus_df = pd.read_csv(corpus_path, encoding='utf-8')

    # путь к базе данных
    db_path = r'datasets/nlp_data.db'

    # открываем базу
    with NlpDatabase(db_path) as db:
        # получаем id текстов, которые уже обработаны
        already_done = db.get_processed_ids('classla')
        print(f"уже обработано ранее: {len(already_done)} текстов")

        # обрабатываем каждый текст
        for idx, row in corpus_df.iterrows():
            text_id = int(row['id'])
            if text_id in already_done:
                continue

            text = str(row['text'])
            title = row.get('title', f'text_{text_id}')

            try:
                # обрабатываем текст через CLASSLA
                sentences, log_info = process_text_classla(text, nlp)

                # конвертируем в плоский список токенов
                flat_tokens = sentences_to_flat_tokens(sentences, text_id=text_id)

                # сохраняем в базу
                db.save_tokens(text_id, flat_tokens, method='classla')
                db.save_log(text_id, 'classla', 'ok',
                           log_info['n_sentences'], log_info['n_tokens'],
                           log_info['elapsed_sec'])
                print(f"  text_id={text_id}: {log_info['n_sentences']} предл., "
                      f"{log_info['n_tokens']} токенов, {log_info['elapsed_sec']:.1f} сек")

            except Exception as e:
                db.save_log(text_id, 'classla', 'error', 0, 0, 0,
                           error=str(e)[:200])
                print(f"  text_id={text_id}: ОШИБКА {e}")
else:
    print("RUN_HEAVY = False, загружаем готовые результаты из базы")
    print("(обработка 98 текстов через CLASSLA заняла ~50 мин)")

### Rule-based: обработка корпуса

Обработка rule-based лемматизатором **не требует повторного запуска CLASSLA**. Мы читаем готовые токены из таблицы `classla_tokens` (form, upos), применяем `rule_based_lemmatize` к каждому токену и сохраняем в `rulebased_tokens`. Это занимает ~33 секунды.

In [ ]:
import sqlite3

if RUN_HEAVY:
    db_path = r'datasets/nlp_data.db'

    with NlpDatabase(db_path) as db:
        # читаем все classla-токены из базы
        raw_conn = sqlite3.connect(db_path)
        cur = raw_conn.cursor()
        cur.execute("""
            SELECT text_id, sentence_id, token_id, form, lemma, upos, xpos
            FROM classla_tokens
            ORDER BY text_id, sentence_id, token_id
        """)
        all_rows = cur.fetchall()
        raw_conn.close()
        print(f'прочитали {len(all_rows):,} токенов из classla_tokens')

        # группируем по text_id
        groups = {}
        for row in all_rows:
            tid = row[0]
            if tid not in groups:
                groups[tid] = []
            groups[tid].append(row)

        # применяем rule_based_lemmatize к каждому токену
        for tid in sorted(groups.keys()):
            rows = groups[tid]
            tokens_rb = []
            for row in rows:
                _, sent_id, tok_id, form, _, upos, xpos = row
                lemma_rb = rule_based_lemmatize(form, upos)
                tokens_rb.append({
                    'sentence_id': sent_id,
                    'token_id': tok_id,
                    'form': form,
                    'lemma': lemma_rb,
                    'upos': upos,
                    'xpos': xpos,
                })
            db.save_tokens(text_id=tid, tokens=tokens_rb, method='rulebased')

        print('rule-based обработка корпуса завершена')
else:
    print("RUN_HEAVY = False, результаты rule-based лемматизации уже в базе")

In [ ]:
# загружаем несколько примеров из обеих таблиц для наглядного сравнения
db_path = r'datasets/nlp_data.db'

with NlpDatabase(db_path) as db:
    # берем первые 15 токенов первого текста
    cl_tokens = db.get_tokens(text_id=6, method='classla')[:15]
    rb_tokens = db.get_tokens(text_id=6, method='rulebased')[:15]

    print("Сравнение лемм CLASSLA vs rule-based (text_id=6, первые 15 токенов):")
    print(f"  {'форма':<20s} {'POS':<6s} {'CLASSLA':<20s} {'rule-based':<20s}")
    print(f"  {'-'*20} {'-'*6} {'-'*20} {'-'*20}")
    for cl, rb in zip(cl_tokens, rb_tokens):
        marker = '' if cl['lemma'] == rb['lemma'] else ' <--'
        print(f"  {cl['form']:<20s} {cl['upos']:<6s} {cl['lemma']:<20s} {rb['lemma']:<20s}{marker}")

## 2.7. Gold Standard для оценки лемматизации

Для объективного сравнения двух лемматизаторов нужен **gold standard** — набор данных с вручную проверенными леммами.

**UD Macedonian-MTB** (Universal Dependencies): ~155 предложений, ~1360 токенов с ручной разметкой (леммы, POS, морфологические признаки). Лицензия CC BY-SA 4.0.

Ограничение: UD-MTB — новостные/учебные тексты, а наш корпус — художественная литература. Тем не менее, это единственный доступный gold standard для македонского.

In [ ]:
import os
import urllib.request
import csv
from collections import Counter

# ставим UTF-8 для корректного вывода кириллицы в Windows-консоли

# добавляем путь к temp, чтобы импортировать NlpDatabase

# импортируем класс для работы с SQLite-базой NLP-данных


# путь к базе данных проекта
DB_PATH = r'C:\Projects\makedonian-course\datasets\nlp_data.db'

# путь для экспорта CSV
CSV_PATH = r'C:\Projects\makedonian-course\datasets\gold_standard_lemmas.csv'

# URL для скачивания UD Macedonian-MTB (test split)
UD_MTB_URL = 'https://raw.githubusercontent.com/UniversalDependencies/UD_Macedonian-MTB/master/mk_mtb-ud-test.conllu'

# список URL для SETimes.MK (пробуем по очереди)
SETIMES_URLS = [
    'https://www.clarin.si/repository/xmlui/bitstream/handle/11356/1843/SETimes.MK.conllu',
    'https://www.clarin.si/repository/xmlui/bitstream/handle/11356/1843/SETimes.MK-test.conllu',
]

# таймаут на скачивание в секундах
DOWNLOAD_TIMEOUT = 30


def parse_conllu(text, source_name):
    """
    парсит текст в формате CoNLL-U и возвращает список словарей для gold_standard
    text — строка с содержимым CoNLL-U файла
    source_name — название источника (например, 'UD-MTB' или 'SETimes')
    возвращает список словарей с ключами: source, sentence_id, token_id, form, lemma_gold, upos
    """
    # список для результатов
    records = []
    # счетчик предложений, начинаем с нуля
    sentence_id = 0
    # флаг: встретили ли хотя бы один токен в текущем предложении
    has_tokens = False

    # разбиваем текст на строки
    lines = text.split('\n')

    for line in lines:
        # убираем пробелы по краям
        line = line.strip()

        # пустая строка означает конец предложения
        if line == '':
            if has_tokens:
                # переходим к следующему предложению
                sentence_id += 1
                # сбрасываем флаг
                has_tokens = False
            continue

        # строки с # — комментарии CoNLL-U, пропускаем
        if line.startswith('#'):
            continue

        # разбиваем строку по TAB (CoNLL-U всегда использует TAB)
        fields = line.split('\t')

        # в CoNLL-U должно быть ровно 10 полей
        if len(fields) != 10:
            continue

        # первое поле — ID токена
        token_id_str = fields[0]

        # пропускаем multiword tokens (ID содержит дефис, например "1-2")
        if '-' in token_id_str:
            continue

        # пропускаем empty nodes (ID содержит точку, например "1.1")
        if '.' in token_id_str:
            continue

        # извлекаем нужные поля
        # fields[0] = ID, fields[1] = FORM, fields[2] = LEMMA, fields[3] = UPOS
        token_id = int(token_id_str)
        form = fields[1]
        lemma = fields[2]
        upos = fields[3]

        # собираем словарь записи
        record = {
            'source': source_name,
            'sentence_id': sentence_id,
            'token_id': token_id,
            'form': form,
            'lemma_gold': lemma,
            'upos': upos,
        }

        # добавляем в результат
        records.append(record)
        # отмечаем, что в текущем предложении есть токены
        has_tokens = True

    return records


def download_file(url, timeout=DOWNLOAD_TIMEOUT):
    """
    скачивает файл по URL и возвращает его содержимое как строку
    при ошибке возвращает None
    """
    print(f"  скачиваем: {url}")
    try:
        # создаем запрос с User-Agent, чтобы сервер не отклонил
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        # скачиваем с таймаутом
        with urllib.request.urlopen(req, timeout=timeout) as response:
            # читаем байты и декодируем в UTF-8
            raw_bytes = response.read()
            text = raw_bytes.decode('utf-8')
            print(f"  скачали {len(raw_bytes)} байт")
            return text
    except Exception as e:
        # при любой ошибке сети — логируем и возвращаем None
        print(f"  ошибка при скачивании: {e}")
        return None


def print_stats(records, source_name):
    """
    выводит статистику по распарсенным записям gold standard
    records — список словарей из parse_conllu
    """
    if not records:
        print(f"  [{source_name}] записей нет")
        return

    # считаем количество уникальных предложений
    sentence_ids = set(r['sentence_id'] for r in records)
    n_sentences = len(sentence_ids)

    # общее количество токенов
    n_tokens = len(records)

    # количество уникальных лемм
    unique_lemmas = set(r['lemma_gold'] for r in records)
    n_lemmas = len(unique_lemmas)

    # распределение по UPOS-тегам
    upos_counts = Counter(r['upos'] for r in records)

    print(f"  [{source_name}] предложений: {n_sentences}, токенов: {n_tokens}, уникальных лемм: {n_lemmas}")
    print(f"  распределение по UPOS (топ-15):")

    # выводим UPOS-теги, отсортированные по частоте
    for upos, count in upos_counts.most_common(15):
        # доля от общего числа токенов в процентах
        pct = 100.0 * count / n_tokens
        print(f"    {upos:10s} {count:6d}  ({pct:5.1f}%)")


def load_ud_mtb():
    """
    скачивает и парсит UD Macedonian-MTB (test split)
    возвращает список записей или пустой список при ошибке
    """
    print("\n[шаг 5.2] загрузка UD Macedonian-MTB...")

    # скачиваем CoNLL-U файл
    text = download_file(UD_MTB_URL)

    if text is None:
        print("  не удалось скачать UD-MTB, продолжаем без него")
        return []

    # парсим CoNLL-U текст
    records = parse_conllu(text, source_name='UD-MTB')
    print(f"  распарсили {len(records)} записей")

    # выводим статистику
    print_stats(records, 'UD-MTB')

    return records


def load_setimes():
    """
    пробует скачать и распарсить SETimes.MK корпус
    пробует несколько URL, при неудаче возвращает пустой список
    """
    print("\n[шаг 5.3] загрузка SETimes.MK...")

    # пробуем URL-ы по очереди
    for url in SETIMES_URLS:
        text = download_file(url)
        if text is not None:
            # проверяем, что скачанный файл похож на CoNLL-U (содержит TAB-разделенные строки)
            # берем первые 20 непустых не-комментарных строк
            sample_lines = [
                ln for ln in text.split('\n')[:200]
                if ln.strip() and not ln.startswith('#')
            ]
            # в CoNLL-U строки содержат TAB
            tab_lines = sum(1 for ln in sample_lines if '\t' in ln)

            if tab_lines > 5:
                # похоже на CoNLL-U, парсим
                records = parse_conllu(text, source_name='SETimes')
                if records:
                    print(f"  распарсили {len(records)} записей из {url}")
                    print_stats(records, 'SETimes')
                    return records
                else:
                    print(f"  файл скачался, но записей после парсинга нет")
            else:
                print(f"  скачанный файл не похож на CoNLL-U (мало TAB-строк)")

    # ни один URL не сработал
    print("  SETimes.MK не удалось загрузить ни из одного источника")
    print("  продолжаем только с UD-MTB")
    return []


def save_to_db(all_records):
    """
    сохраняет записи gold standard в SQLite-базу
    перед вставкой удаляет старые записи с тем же source (идемпотентность)
    """
    print("\n[шаг 5.5] сохранение в SQLite...")

    # собираем уникальные source-ы из записей
    sources = set(r['source'] for r in all_records)
    print(f"  источники для сохранения: {sources}")

    # открываем базу
    with NlpDatabase(DB_PATH) as db:
        cur = db.conn.cursor()

        # для каждого источника удаляем старые записи (идемпотентность)
        for source in sources:
            cur.execute("DELETE FROM gold_standard WHERE source = ?", (source,))
            deleted = cur.rowcount
            if deleted > 0:
                print(f"  удалили {deleted} старых записей для source='{source}'")

        # фиксируем удаление
        db.conn.commit()

        # сохраняем новые записи через метод NlpDatabase
        db.save_gold_standard(all_records)
        print(f"  сохранили {len(all_records)} записей в gold_standard")

        # проверяем итоговое состояние таблицы
        cur.execute("SELECT source, COUNT(*) as cnt FROM gold_standard GROUP BY source")
        rows = cur.fetchall()
        print("\n  итого в таблице gold_standard:")
        total = 0
        for row in rows:
            print(f"    {row['source']:15s} {row['cnt']:7d} записей")
            total += row['cnt']
        print(f"    {'всего':15s} {total:7d} записей")


def export_to_csv(all_records):
    """
    экспортирует записи gold standard в CSV файл
    """
    print(f"\n[экспорт] сохраняем CSV: {CSV_PATH}")

    # создаем папку, если ее нет
    os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)

    # имена столбцов для CSV
    fieldnames = ['source', 'sentence_id', 'token_id', 'form', 'lemma_gold', 'upos']

    # пишем CSV с UTF-8 BOM для корректного открытия в Excel
    with open(CSV_PATH, 'w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        # записываем заголовок
        writer.writeheader()
        # записываем все строки
        writer.writerows(all_records)

    # размер файла
    size_kb = os.path.getsize(CSV_PATH) / 1024
    print(f"  записали {len(all_records)} строк, размер файла: {size_kb:.1f} КБ")

In [ ]:
# загружаем gold standard
import urllib.request
from collections import Counter

# URL для скачивания UD Macedonian-MTB
UD_MTB_URL = 'https://raw.githubusercontent.com/UniversalDependencies/UD_Macedonian-MTB/master/mk_mtb-ud-test.conllu'

print("Скачиваем UD Macedonian-MTB...")
req = urllib.request.Request(UD_MTB_URL, headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req, timeout=30) as response:
    ud_text = response.read().decode('utf-8')
    print(f"  скачали {len(ud_text)} байт")

# парсим CoNLL-U
ud_records = parse_conllu(ud_text, source_name='UD-MTB')
print(f"  распарсили {len(ud_records)} записей")

# статистика
n_sentences = len(set(r['sentence_id'] for r in ud_records))
n_tokens = len(ud_records)
upos_counts = Counter(r['upos'] for r in ud_records)
print(f"  предложений: {n_sentences}, токенов: {n_tokens}")
print(f"  распределение по POS (топ-10):")
for upos, count in upos_counts.most_common(10):
    print(f"    {upos:10s} {count:5d}  ({100*count/n_tokens:.1f}%)")

# сохраняем в базу
db_path = r'datasets/nlp_data.db'
with NlpDatabase(db_path) as db:
    cur = db.conn.cursor()
    cur.execute("DELETE FROM gold_standard WHERE source = 'UD-MTB'")
    db.conn.commit()
    db.save_gold_standard(ud_records)
    print(f"\nсохранили {len(ud_records)} записей в gold_standard")

## 2.8. Сравнение двух подходов к лемматизации

Сравниваем нейронный (CLASSLA) и rule-based лемматизаторы по трем направлениям:
1. **Accuracy на gold standard** — точность лемматизации на эталонных данных UD-MTB
2. **Анализ ошибок** — какие категории ошибок у каждого подхода
3. **Скорость** — сколько токенов в секунду обрабатывает каждый метод

In [ ]:
import os
import json
import time
import sqlite3
import re

# настраиваем кодировку для Windows

# добавляем temp/ в путь поиска модулей

# импортируем rule-based лемматизатор и функцию поиска в словаре исключений

# путь к базе и папке для результатов
DB_PATH = r'C:\Projects\makedonian-course\datasets\nlp_data.db'
OUTPUT_DIR = r'C:\Projects\makedonian-course\temp'


def load_gold_standard(db_path):
    """
    читаем gold standard из базы, группируем по sentence_id
    возвращаем два объекта:
      - gold_rows: плоский список словарей (для поэлементного сравнения)
      - sentences: dict {sentence_id: [list of token dicts sorted by token_id]}
    """
    # подключаемся к базе
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()

    # читаем все токены gold standard, отсортированные по предложению и позиции
    cur.execute("""
        SELECT source, sentence_id, token_id, form, lemma_gold, upos
        FROM gold_standard
        ORDER BY sentence_id, token_id
    """)
    rows = cur.fetchall()
    conn.close()

    # превращаем sqlite3.Row в обычные словари
    gold_rows = [dict(r) for r in rows]

    # группируем по sentence_id для реконструкции предложений
    sentences = {}
    for row in gold_rows:
        sid = row['sentence_id']
        if sid not in sentences:
            sentences[sid] = []
        sentences[sid].append(row)

    return gold_rows, sentences


def run_classla_on_gold(sentences):
    """
    прогоняем gold standard через CLASSLA в pretokenized-режиме
    sentences: dict {sentence_id: [list of token dicts]}
    возвращаем dict {(sentence_id, token_id): classla_lemma}
    """
    # импортируем CLASSLA здесь, чтобы не грузить при ненадобности
    import classla

    # создаём pipeline с pretokenized=True — CLASSLA не будет перетокенизировать,
    # а только сделает POS-tagging и лемматизацию
    print('загружаем CLASSLA pipeline (pretokenized)...')
    t0 = time.time()
    nlp = classla.Pipeline('mk', processors='tokenize,pos,lemma',
                           tokenize_pretokenized=True, logging_level='WARN')
    print(f'pipeline загружен за {time.time() - t0:.1f} сек')

    # собираем отсортированные sentence_id
    sorted_sids = sorted(sentences.keys())

    # строим текст для pretokenized-режима:
    # каждая строка — одно предложение, токены через пробел
    lines = []
    for sid in sorted_sids:
        tokens = sentences[sid]
        # склеиваем формы через пробел
        line = ' '.join(t['form'] for t in tokens)
        lines.append(line)

    # объединяем строки через перевод строки
    pretok_text = '\n'.join(lines)

    # прогоняем через CLASSLA
    print(f'обрабатываем {len(lines)} предложений через CLASSLA...')
    t0 = time.time()
    doc = nlp(pretok_text)
    elapsed = time.time() - t0
    print(f'CLASSLA обработала за {elapsed:.1f} сек')

    # проверяем, что количество предложений совпало
    if len(doc.sentences) != len(sorted_sids):
        print(f'ВНИМАНИЕ: CLASSLA вернула {len(doc.sentences)} предложений, '
              f'а в gold standard {len(sorted_sids)}')

    # собираем результат: сопоставляем по позиции
    classla_lemmas = {}
    for i, sid in enumerate(sorted_sids):
        gold_tokens = sentences[sid]
        if i >= len(doc.sentences):
            # если CLASSLA вернула меньше предложений — пропускаем
            print(f'  предложение {sid}: нет в выводе CLASSLA')
            continue

        classla_sent = doc.sentences[i]
        classla_words = classla_sent.words

        # сопоставляем токены по позиции
        for j, gold_tok in enumerate(gold_tokens):
            if j < len(classla_words):
                classla_lemmas[(sid, gold_tok['token_id'])] = classla_words[j].lemma
            else:
                # если CLASSLA вернула меньше токенов в предложении
                classla_lemmas[(sid, gold_tok['token_id'])] = None

    return classla_lemmas


def run_rulebased_on_gold(gold_rows):
    """
    прогоняем gold standard через rule-based лемматизатор
    возвращаем dict {(sentence_id, token_id): rb_lemma}
    """
    rb_lemmas = {}
    for row in gold_rows:
        sid = row['sentence_id']
        tid = row['token_id']
        form = row['form']
        upos = row['upos']
        # вызываем rule-based лемматизатор
        lemma = rule_based_lemmatize(form, upos)
        rb_lemmas[(sid, tid)] = lemma
    return rb_lemmas


def compute_accuracy(gold_rows, predicted_lemmas, case_sensitive=True):
    """
    считаем accuracy: % токенов, где predicted == gold
    predicted_lemmas: dict {(sentence_id, token_id): lemma_str}
    case_sensitive: если False — сравниваем в нижнем регистре
    возвращаем (n_correct, n_total, accuracy)
    """
    n_correct = 0
    n_total = 0
    for row in gold_rows:
        key = (row['sentence_id'], row['token_id'])
        pred = predicted_lemmas.get(key)
        if pred is None:
            # пропущенный токен считаем ошибкой
            n_total += 1
            continue
        gold = row['lemma_gold']
        if not case_sensitive:
            pred = pred.lower()
            gold = gold.lower()
        if pred == gold:
            n_correct += 1
        n_total += 1
    accuracy = n_correct / n_total if n_total > 0 else 0.0
    return n_correct, n_total, accuracy


def accuracy_by_pos(gold_rows, predicted_lemmas, case_sensitive=True):
    """
    считаем accuracy отдельно по каждому POS-тегу
    возвращаем dict {upos: (n_correct, n_total, accuracy)}
    """
    # собираем статистику по POS
    pos_stats = {}
    for row in gold_rows:
        upos = row['upos']
        if upos not in pos_stats:
            pos_stats[upos] = {'correct': 0, 'total': 0}
        key = (row['sentence_id'], row['token_id'])
        pred = predicted_lemmas.get(key)
        gold = row['lemma_gold']
        if pred is None:
            pos_stats[upos]['total'] += 1
            continue
        if not case_sensitive:
            pred = pred.lower()
            gold = gold.lower()
        if pred == gold:
            pos_stats[upos]['correct'] += 1
        pos_stats[upos]['total'] += 1

    # превращаем в удобный формат
    result = {}
    for upos, stats in pos_stats.items():
        c = stats['correct']
        t = stats['total']
        acc = c / t if t > 0 else 0.0
        result[upos] = (c, t, acc)
    return result


def error_analysis(gold_rows, classla_lemmas, rb_lemmas):
    """
    шаг 7.2: анализ ошибок
    находим токены, где:
      - rule-based ошибся, CLASSLA правильно
      - CLASSLA ошибся, rule-based правильно
      - оба ошиблись
    возвращаем три списка словарей с подробностями
    """
    rb_wrong_cl_right = []
    cl_wrong_rb_right = []
    both_wrong = []

    for row in gold_rows:
        key = (row['sentence_id'], row['token_id'])
        gold = row['lemma_gold'].lower()
        form = row['form']
        upos = row['upos']

        cl_pred = classla_lemmas.get(key)
        rb_pred = rb_lemmas.get(key)

        # приводим к нижнему регистру для сравнения
        cl_ok = (cl_pred is not None) and (cl_pred.lower() == gold)
        rb_ok = (rb_pred is not None) and (rb_pred.lower() == gold)

        if not rb_ok and cl_ok:
            # rule-based ошибся, CLASSLA правильно
            rb_wrong_cl_right.append({
                'form': form, 'upos': upos, 'gold': row['lemma_gold'],
                'rb_pred': rb_pred, 'cl_pred': cl_pred,
                'sentence_id': row['sentence_id'], 'token_id': row['token_id']
            })
        elif not cl_ok and rb_ok:
            # CLASSLA ошибся, rule-based правильно
            cl_wrong_rb_right.append({
                'form': form, 'upos': upos, 'gold': row['lemma_gold'],
                'rb_pred': rb_pred, 'cl_pred': cl_pred,
                'sentence_id': row['sentence_id'], 'token_id': row['token_id']
            })
        elif not cl_ok and not rb_ok:
            # оба ошиблись
            both_wrong.append({
                'form': form, 'upos': upos, 'gold': row['lemma_gold'],
                'rb_pred': rb_pred, 'cl_pred': cl_pred,
                'sentence_id': row['sentence_id'], 'token_id': row['token_id']
            })

    return rb_wrong_cl_right, cl_wrong_rb_right, both_wrong


def classify_rb_error(row):
    """
    классифицируем ошибку rule-based лемматизатора по категориям:
    1. article — ошибка снятия артикля (сняли лишнее или не сняли)
    2. verb — ошибка глагольной формы
    3. adj — ошибка прилагательного
    4. irregular — неправильная/супплетивная форма, нет в словаре исключений
    5. ambiguity — омонимия (одна форма, несколько возможных лемм)
    6. oov_rules — слово не в словаре, правила не помогли
    7. propn_foreign — имя собственное или иностранное слово
    """
    form = row['form']
    upos = row['upos']
    gold = row['gold']
    pred = row['rb_pred']
    form_lower = form.lower()

    # имена собственные
    if upos == 'PROPN':
        return 'propn_foreign'

    # латиница — иностранное слово
    if re.fullmatch(r'[a-zA-Z]+', form):
        return 'propn_foreign'

    # проверяем, есть ли слово в словаре исключений
    exc = lookup_exception(form_lower, upos)

    # глаголы
    if upos in ('VERB', 'AUX'):
        if exc is not None and exc != gold.lower():
            # есть в словаре, но словарь даёт не ту лемму — нетипично
            return 'irregular'
        return 'verb'

    # прилагательные
    if upos == 'ADJ':
        return 'adj'

    # существительные — часто ошибки артикля
    if upos == 'NOUN':
        # если gold и pred отличаются на типичный суффикс артикля
        article_suffixes = ['от', 'та', 'то', 'ов', 'ва', 'во', 'он', 'на', 'но', 'те', 'ве', 'не']
        # проверяем: если gold + суффикс == form_lower — значит артикль не сняли
        for suf in article_suffixes:
            if form_lower == gold.lower() + suf:
                if pred == form_lower:
                    return 'article'
        # проверяем: если pred короче gold — значит сняли лишнее
        if len(pred) < len(gold.lower()):
            return 'article'
        return 'oov_rules'

    # местоимения и определители
    if upos in ('PRON', 'DET'):
        if exc is None:
            return 'irregular'
        return 'oov_rules'

    # все остальное
    if exc is not None and exc != gold.lower():
        return 'irregular'

    return 'oov_rules'


def coverage_analysis(gold_rows, rb_lemmas):
    """
    шаг 7.3: анализ покрытия
    считаем:
      - exception_coverage: % токенов, найденных в словаре исключений
      - rule_activation: % токенов, где rule-based изменил форму (lemma != form.lower())
      - oov_accuracy: accuracy на словах НЕ из словаря исключений
      - known_accuracy: accuracy на словах ИЗ словаря исключений
    """
    n_in_dict = 0
    n_total = 0
    n_rule_changed = 0

    # для accuracy по группам
    known_correct = 0
    known_total = 0
    oov_correct = 0
    oov_total = 0

    for row in gold_rows:
        form = row['form']
        upos = row['upos']
        gold = row['lemma_gold'].lower()
        form_lower = form.lower()

        # пропускаем пунктуацию — она тривиальна
        if upos == 'PUNCT':
            continue

        n_total += 1

        # проверяем, есть ли форма в словаре исключений
        exc = lookup_exception(form_lower, upos)
        in_dict = exc is not None

        if in_dict:
            n_in_dict += 1

        # проверяем, изменил ли rule-based форму
        key = (row['sentence_id'], row['token_id'])
        rb_pred = rb_lemmas.get(key)
        if rb_pred is not None and rb_pred != form_lower:
            n_rule_changed += 1

        # accuracy по группам
        rb_ok = (rb_pred is not None) and (rb_pred.lower() == gold)
        if in_dict:
            known_total += 1
            if rb_ok:
                known_correct += 1
        else:
            oov_total += 1
            if rb_ok:
                oov_correct += 1

    exception_coverage = n_in_dict / n_total if n_total > 0 else 0.0
    rule_activation = n_rule_changed / n_total if n_total > 0 else 0.0
    known_accuracy = known_correct / known_total if known_total > 0 else 0.0
    oov_accuracy = oov_correct / oov_total if oov_total > 0 else 0.0

    return {
        'exception_coverage': exception_coverage,
        'n_in_dict': n_in_dict,
        'n_total_non_punct': n_total,
        'rule_activation': rule_activation,
        'n_rule_changed': n_rule_changed,
        'known_accuracy': known_accuracy,
        'known_correct': known_correct,
        'known_total': known_total,
        'oov_accuracy': oov_accuracy,
        'oov_correct': oov_correct,
        'oov_total': oov_total,
    }


def main():
    print('=' * 70)
    print('ОЦЕНКА ЛЕММАТИЗАТОРОВ НА GOLD STANDARD (UD Macedonian-MTB)')
    print('=' * 70)
    print()

    # загружаем gold standard
    print('загружаем gold standard из базы...')
    gold_rows, sentences = load_gold_standard(DB_PATH)
    print(f'  токенов: {len(gold_rows)}, предложений: {len(sentences)}')
    print()

    # запускаем CLASSLA на gold standard
    classla_lemmas = run_classla_on_gold(sentences)
    print(f'  CLASSLA обработала {len(classla_lemmas)} токенов')
    print()

    # запускаем rule-based на gold standard
    print('обрабатываем gold standard rule-based лемматизатором...')
    rb_lemmas = run_rulebased_on_gold(gold_rows)
    print(f'  rule-based обработал {len(rb_lemmas)} токенов')
    print()

    # шаг 7.1: accuracy
    print('=' * 70)
    print('ШАГ 7.1: ACCURACY НА GOLD STANDARD')
    print('=' * 70)
    print()

    # exact match accuracy (case-sensitive)
    cl_correct, cl_total, cl_acc = compute_accuracy(gold_rows, classla_lemmas, case_sensitive=True)
    rb_correct, rb_total, rb_acc = compute_accuracy(gold_rows, rb_lemmas, case_sensitive=True)
    print(f'Exact Match Accuracy (case-sensitive):')
    print(f'  CLASSLA:    {cl_correct}/{cl_total} = {cl_acc:.4f} ({cl_acc*100:.1f}%)')
    print(f'  Rule-based: {rb_correct}/{rb_total} = {rb_acc:.4f} ({rb_acc*100:.1f}%)')
    print()

    # case-insensitive accuracy
    cl_ci_correct, cl_ci_total, cl_ci_acc = compute_accuracy(gold_rows, classla_lemmas, case_sensitive=False)
    rb_ci_correct, rb_ci_total, rb_ci_acc = compute_accuracy(gold_rows, rb_lemmas, case_sensitive=False)
    print(f'Case-insensitive Accuracy:')
    print(f'  CLASSLA:    {cl_ci_correct}/{cl_ci_total} = {cl_ci_acc:.4f} ({cl_ci_acc*100:.1f}%)')
    print(f'  Rule-based: {rb_ci_correct}/{rb_ci_total} = {rb_ci_acc:.4f} ({rb_ci_acc*100:.1f}%)')
    print()

    # accuracy by POS
    print(f'Accuracy by POS (case-insensitive):')
    cl_pos = accuracy_by_pos(gold_rows, classla_lemmas, case_sensitive=False)
    rb_pos = accuracy_by_pos(gold_rows, rb_lemmas, case_sensitive=False)

    # собираем все POS-теги и сортируем по количеству токенов (больше -> первый)
    all_pos = sorted(set(cl_pos.keys()) | set(rb_pos.keys()),
                     key=lambda p: -(cl_pos.get(p, (0, 0, 0))[1]))

    # заголовок таблицы
    print(f'  {"POS":<8s} {"Count":>6s} {"CLASSLA":>10s} {"Rule-based":>12s} {"Diff":>8s}')
    print(f'  {"-"*8} {"-"*6} {"-"*10} {"-"*12} {"-"*8}')

    # строки таблицы
    for pos in all_pos:
        cl_c, cl_t, cl_a = cl_pos.get(pos, (0, 0, 0.0))
        rb_c, rb_t, rb_a = rb_pos.get(pos, (0, 0, 0.0))
        count = max(cl_t, rb_t)
        # diff в процентных пунктах (cl_a и rb_a уже доли от 0 до 1)
        diff_pp = (cl_a - rb_a) * 100
        diff_str = f'{diff_pp:+.1f}pp' if abs(diff_pp) > 0.05 else '0.0pp'
        print(f'  {pos:<8s} {count:>6d} {cl_a*100:>9.1f}% {rb_a*100:>11.1f}% {diff_str:>8s}')

    print()

    # шаг 7.2: анализ ошибок
    print('=' * 70)
    print('ШАГ 7.2: АНАЛИЗ ОШИБОК')
    print('=' * 70)
    print()

    rb_wrong_cl_right, cl_wrong_rb_right, both_wrong = error_analysis(
        gold_rows, classla_lemmas, rb_lemmas)

    print(f'Rule-based ошибся, CLASSLA правильно: {len(rb_wrong_cl_right)} токенов')
    print(f'CLASSLA ошибся, rule-based правильно: {len(cl_wrong_rb_right)} токенов')
    print(f'Оба ошиблись:                         {len(both_wrong)} токенов')
    print()

    # примеры: rule-based wrong, classla right (до 15)
    print(f'Примеры (rule-based wrong, CLASSLA right) [до 15]:')
    print(f'  {"form":<18s} {"upos":<7s} {"gold":<15s} {"rb_pred":<15s} {"cl_pred":<15s}')
    print(f'  {"-"*18} {"-"*7} {"-"*15} {"-"*15} {"-"*15}')
    for item in rb_wrong_cl_right[:15]:
        print(f'  {item["form"]:<18s} {item["upos"]:<7s} {item["gold"]:<15s} '
              f'{item["rb_pred"]:<15s} {item["cl_pred"]:<15s}')
    print()

    # примеры: classla wrong, rule-based right (до 15)
    print(f'Примеры (CLASSLA wrong, rule-based right) [до 15]:')
    print(f'  {"form":<18s} {"upos":<7s} {"gold":<15s} {"rb_pred":<15s} {"cl_pred":<15s}')
    print(f'  {"-"*18} {"-"*7} {"-"*15} {"-"*15} {"-"*15}')
    for item in cl_wrong_rb_right[:15]:
        print(f'  {item["form"]:<18s} {item["upos"]:<7s} {item["gold"]:<15s} '
              f'{item["rb_pred"]:<15s} {item["cl_pred"]:<15s}')
    print()

    # примеры: оба ошиблись (до 15)
    print(f'Примеры (оба ошиблись) [до 15]:')
    print(f'  {"form":<18s} {"upos":<7s} {"gold":<15s} {"rb_pred":<15s} {"cl_pred":<15s}')
    print(f'  {"-"*18} {"-"*7} {"-"*15} {"-"*15} {"-"*15}')
    for item in both_wrong[:15]:
        print(f'  {item["form"]:<18s} {item["upos"]:<7s} {item["gold"]:<15s} '
              f'{item["rb_pred"]:<15s} {item["cl_pred"]:<15s}')
    print()

    # классификация ошибок rule-based по категориям
    # берём все токены, где rule-based ошибся (и classla right, и both wrong)
    all_rb_errors = rb_wrong_cl_right + both_wrong
    print(f'Классификация ошибок rule-based ({len(all_rb_errors)} ошибок):')
    print()

    # считаем категории
    category_counts = {}
    category_examples = {}
    for item in all_rb_errors:
        cat = classify_rb_error(item)
        if cat not in category_counts:
            category_counts[cat] = 0
            category_examples[cat] = []
        category_counts[cat] += 1
        # сохраняем до 5 примеров
        if len(category_examples[cat]) < 5:
            category_examples[cat].append(item)

    # описания категорий
    cat_descriptions = {
        'article': 'ошибка снятия артикля (сняли лишнее или не сняли)',
        'verb': 'ошибка глагольной формы (не то окончание/основа)',
        'adj': 'ошибка прилагательного (не привели к м.р.)',
        'irregular': 'неправильная/супплетивная форма, нет в словаре исключений',
        'ambiguity': 'омонимия (одна форма, несколько возможных лемм)',
        'oov_rules': 'слово не в словаре, правила не помогли',
        'propn_foreign': 'имя собственное или иностранное слово',
    }

    # выводим по категориям, сортируя по количеству ошибок
    for cat, count in sorted(category_counts.items(), key=lambda x: -x[1]):
        desc = cat_descriptions.get(cat, cat)
        print(f'  {cat} ({count} ошибок) -- {desc}')
        for ex in category_examples[cat]:
            print(f'    {ex["form"]:<18s} {ex["upos"]:<7s} gold={ex["gold"]:<15s} '
                  f'rb={ex["rb_pred"]:<15s}')
        print()

    # шаг 7.3: покрытие
    print('=' * 70)
    print('ШАГ 7.3: ПОКРЫТИЕ И OOV-АНАЛИЗ')
    print('=' * 70)
    print()

    cov = coverage_analysis(gold_rows, rb_lemmas)

    print(f'Exception Dict Coverage (без PUNCT):')
    print(f'  токенов в словаре: {cov["n_in_dict"]} из {cov["n_total_non_punct"]} '
          f'= {cov["exception_coverage"]*100:.1f}%')
    print()

    print(f'Rule Activation Rate (без PUNCT):')
    print(f'  правила изменили форму: {cov["n_rule_changed"]} из {cov["n_total_non_punct"]} '
          f'= {cov["rule_activation"]*100:.1f}%')
    print()

    print(f'Known-word Accuracy (rule-based, слова из словаря):')
    print(f'  {cov["known_correct"]}/{cov["known_total"]} = {cov["known_accuracy"]*100:.1f}%')
    print()

    # разбираем, почему known-word accuracy низкая
    # основная причина: разное соглашение о леммах местоимений
    # UD gold standard: "го" -> "го", "се" -> "се", "ја" -> "ја" (форма = лемма)
    # наш словарь: "го" -> "тој", "се" -> "себе", "ја" -> "таа" (полные местоимения)
    pronoun_mismatch = 0
    other_known_errors = 0
    for row in gold_rows:
        if row['upos'] == 'PUNCT':
            continue
        exc = lookup_exception(row['form'].lower(), row['upos'])
        if exc is None:
            continue
        key = (row['sentence_id'], row['token_id'])
        rb_pred = rb_lemmas.get(key, '')
        gold = row['lemma_gold'].lower()
        if rb_pred.lower() != gold:
            if row['upos'] == 'PRON':
                pronoun_mismatch += 1
            else:
                other_known_errors += 1

    print(f'  из них ошибки на PRON (разное соглашение о леммах): {pronoun_mismatch}')
    print(f'  остальные ошибки на known-словах: {other_known_errors}')
    print(f'  в UD gold standard краткие местоимения "го","се","ја" = сами себе леммы,')
    print(f'  а наш словарь маппит их на полные формы "тој","себе","таа"')
    print()

    print(f'OOV Accuracy (rule-based, слова НЕ из словаря):')
    print(f'  {cov["oov_correct"]}/{cov["oov_total"]} = {cov["oov_accuracy"]*100:.1f}%')
    print()

    # сохраняем метрики в JSON для шага 7.5
    metrics = {
        'gold_standard': {
            'n_tokens': len(gold_rows),
            'n_sentences': len(sentences),
            'source': 'UD-MTB',
        },
        'classla': {
            'exact_match_accuracy': round(cl_acc, 4),
            'case_insensitive_accuracy': round(cl_ci_acc, 4),
            'correct': cl_correct,
            'total': cl_total,
            'accuracy_by_pos': {
                pos: {'correct': c, 'total': t, 'accuracy': round(a, 4)}
                for pos, (c, t, a) in cl_pos.items()
            },
        },
        'rulebased': {
            'exact_match_accuracy': round(rb_acc, 4),
            'case_insensitive_accuracy': round(rb_ci_acc, 4),
            'correct': rb_correct,
            'total': rb_total,
            'accuracy_by_pos': {
                pos: {'correct': c, 'total': t, 'accuracy': round(a, 4)}
                for pos, (c, t, a) in rb_pos.items()
            },
        },
        'error_analysis': {
            'rb_wrong_cl_right': len(rb_wrong_cl_right),
            'cl_wrong_rb_right': len(cl_wrong_rb_right),
            'both_wrong': len(both_wrong),
            'rb_error_categories': {
                cat: count for cat, count in sorted(category_counts.items(), key=lambda x: -x[1])
            },
        },
        'coverage': cov,
    }

    # путь к JSON-файлу с метриками
    metrics_path = os.path.join(OUTPUT_DIR, 'step7_1_3_metrics.json')
    with open(metrics_path, 'w', encoding='utf-8') as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)
    print(f'метрики сохранены в {metrics_path}')
    print()

    print('=' * 70)
    print('ГОТОВО')
    print('=' * 70)

In [ ]:
# запускаем оценку на gold standard
import json

db_path = r'datasets/nlp_data.db'

# загружаем gold standard из базы
print('загружаем gold standard...')
gold_rows, sentences = load_gold_standard(db_path)
print(f'  токенов: {len(gold_rows)}, предложений: {len(sentences)}')

# прогоняем через CLASSLA
classla_lemmas = run_classla_on_gold(sentences)
print(f'  CLASSLA обработала {len(classla_lemmas)} токенов')

# прогоняем через rule-based
rb_lemmas = run_rulebased_on_gold(gold_rows)
print(f'  rule-based обработал {len(rb_lemmas)} токенов')

# accuracy
print('\n--- ACCURACY ---')
cl_c, cl_t, cl_acc = compute_accuracy(gold_rows, classla_lemmas, case_sensitive=True)
rb_c, rb_t, rb_acc = compute_accuracy(gold_rows, rb_lemmas, case_sensitive=True)
print(f'Exact Match: CLASSLA {cl_acc:.1%}, Rule-based {rb_acc:.1%}')

cl_ci_c, cl_ci_t, cl_ci_acc = compute_accuracy(gold_rows, classla_lemmas, case_sensitive=False)
rb_ci_c, rb_ci_t, rb_ci_acc = compute_accuracy(gold_rows, rb_lemmas, case_sensitive=False)
print(f'Case-insensitive: CLASSLA {cl_ci_acc:.1%}, Rule-based {rb_ci_acc:.1%}')

# accuracy by POS
print('\n--- ACCURACY BY POS ---')
cl_pos = accuracy_by_pos(gold_rows, classla_lemmas, case_sensitive=False)
rb_pos = accuracy_by_pos(gold_rows, rb_lemmas, case_sensitive=False)

all_pos_tags = sorted(cl_pos.keys(), key=lambda p: -cl_pos[p][1])
print(f'  {"POS":<8s} {"Count":>6s} {"CLASSLA":>10s} {"Rule-based":>12s}')
for pos in all_pos_tags:
    cl_a = cl_pos.get(pos, (0,0,0))[2]
    rb_a = rb_pos.get(pos, (0,0,0))[2]
    count = cl_pos.get(pos, (0,0,0))[1]
    print(f'  {pos:<8s} {count:>6d} {cl_a:>9.1%} {rb_a:>11.1%}')

# анализ ошибок
print('\n--- АНАЛИЗ ОШИБОК ---')
rb_wrong_cl_right, cl_wrong_rb_right, both_wrong = error_analysis(
    gold_rows, classla_lemmas, rb_lemmas)
print(f'Rule-based ошибся, CLASSLA правильно: {len(rb_wrong_cl_right)}')
print(f'CLASSLA ошибся, rule-based правильно: {len(cl_wrong_rb_right)}')
print(f'Оба ошиблись: {len(both_wrong)}')

# покрытие
print('\n--- ПОКРЫТИЕ ---')
cov = coverage_analysis(gold_rows, rb_lemmas)
print(f'Exception Dict Coverage: {cov["exception_coverage"]:.1%}')
print(f'Rule Activation Rate: {cov["rule_activation"]:.1%}')
print(f'OOV Accuracy: {cov["oov_accuracy"]:.1%}')

In [ ]:
# загружаем метрики из JSON (бенчмарк скорости уже был проведен)
metrics_path = r'temp/step7_1_3_metrics.json'
with open(metrics_path, 'r', encoding='utf-8') as f:
    metrics = json.load(f)

# скорость
print('--- СКОРОСТЬ ---')
cl_speed = metrics['speed']['classla']['tokens_per_sec']
rb_speed = metrics['speed']['rulebased']['tokens_per_sec']
speedup = metrics['speed']['speedup']
print(f'CLASSLA: {cl_speed:,.0f} токенов/сек')
print(f'Rule-based: {rb_speed:,.0f} токенов/сек')
print(f'Ускорение: {speedup:.0f}x')

# сводная таблица
print('\n--- СВОДНАЯ ТАБЛИЦА ---')
print(f'  {"Метрика":<35s} {"CLASSLA":>15s} {"Rule-based":>15s}')
print(f'  {"-"*35} {"-"*15} {"-"*15}')
print(f'  {"Accuracy (case-insensitive)":<35s} {"85.9%":>15s} {"73.6%":>15s}')
print(f'  {"Accuracy (exact match)":<35s} {"84.8%":>15s} {"72.5%":>15s}')
print(f'  {"Accuracy (NOUN)":<35s} {"98.3%":>15s} {"68.7%":>15s}')
print(f'  {"Accuracy (VERB)":<35s} {"82.3%":>15s} {"59.1%":>15s}')
print(f'  {"Accuracy (ADJ)":<35s} {"90.9%":>15s} {"50.0%":>15s}')
print(f'  {"Скорость (токенов/сек)":<35s} {"113":>15s} {"215,963":>15s}')
print(f'  {"Ускорение":<35s} {"-":>15s} {"1,911x":>15s}')

## 2.9. Визуализация результатов сравнения

In [ ]:
# настройка matplotlib для кириллицы
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# цветовая палитра: CLASSLA = синий, rule-based = оранжевый
COLOR_CLASSLA = '#3274A1'
COLOR_RULEBASED = '#E1812C'

# подбираем шрифт с поддержкой кириллицы
for font_name in ['DejaVu Sans', 'Arial', 'Tahoma']:
    if font_name in {f.name for f in matplotlib.font_manager.fontManager.ttflist}:
        plt.rcParams['font.family'] = 'sans-serif'
        plt.rcParams['font.sans-serif'] = [font_name]
        print(f'шрифт: {font_name}')
        break

In [ ]:
import os
import json

# настраиваем кодировку stdout для Windows

# headless-рендеринг: без GUI, чтобы работало на сервере и в CI
import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# добавляем temp/ в путь, чтобы импортировать rule-based лемматизатор и функции оценки

# абсолютные пути к ключевым файлам и папкам проекта
BASE_DIR = r'C:\Projects\makedonian-course'
METRICS_PATH = os.path.join(BASE_DIR, 'temp', 'step7_1_3_metrics.json')
VIS_DIR = os.path.join(BASE_DIR, 'temp', 'visualization')
DATASETS_DIR = os.path.join(BASE_DIR, 'datasets')
DB_PATH = os.path.join(BASE_DIR, 'datasets', 'nlp_data.db')

# цветовая палитра: CLASSLA = синий, rule-based = оранжевый
COLOR_CLASSLA = '#3274A1'
COLOR_RULEBASED = '#E1812C'

# DPI для сохранения картинок
DPI = 150


def setup_font():
    """
    подбираем шрифт, который поддерживает кириллицу
    пробуем DejaVu Sans, потом Arial, потом Tahoma
    """
    # список кандидатов в порядке приоритета
    candidates = ['DejaVu Sans', 'Arial', 'Tahoma']

    # получаем список доступных шрифтов в системе
    available = set()
    for f in matplotlib.font_manager.fontManager.ttflist:
        available.add(f.name)

    # ищем первый подходящий
    for name in candidates:
        if name in available:
            # нашли — устанавливаем глобально
            plt.rcParams['font.family'] = 'sans-serif'
            plt.rcParams['font.sans-serif'] = [name]
            print(f'шрифт: {name}')
            return name

    # если ничего не нашли, оставляем matplotlib по умолчанию
    print('подходящий кириллический шрифт не найден, используем matplotlib default')
    return None


def load_metrics():
    """
    загружаем метрики из JSON-файла, созданного на шаге 7
    """
    with open(METRICS_PATH, 'r', encoding='utf-8') as f:
        metrics = json.load(f)
    print(f'метрики загружены из {METRICS_PATH}')
    return metrics


def plot_accuracy_by_pos(metrics):
    """
    8.1: столбчатая диаграмма accuracy по POS-тегам
    два ряда столбцов (CLASSLA и rule-based) для каждого POS
    POS-теги отсортированы по total (убывание) — частые слева
    """
    # вытаскиваем accuracy по POS для обоих лемматизаторов
    cl_pos = metrics['classla']['accuracy_by_pos']
    rb_pos = metrics['rulebased']['accuracy_by_pos']

    # собираем все POS-теги и сортируем по total (убывание)
    all_pos = sorted(cl_pos.keys(), key=lambda p: -cl_pos[p]['total'])

    # готовим данные для графика
    pos_labels = all_pos
    cl_accs = [cl_pos[p]['accuracy'] * 100 for p in all_pos]
    rb_accs = [rb_pos[p]['accuracy'] * 100 for p in all_pos]
    totals = [cl_pos[p]['total'] for p in all_pos]

    # позиции столбцов по оси X
    x = np.arange(len(pos_labels))
    # ширина одного столбца
    bar_width = 0.35

    # создаём фигуру
    fig, ax = plt.subplots(figsize=(14, 6))

    # рисуем столбцы CLASSLA (смещение влево на половину ширины)
    bars_cl = ax.bar(x - bar_width / 2, cl_accs, bar_width,
                     label='CLASSLA', color=COLOR_CLASSLA, edgecolor='white', linewidth=0.5)

    # рисуем столбцы rule-based (смещение вправо на половину ширины)
    bars_rb = ax.bar(x + bar_width / 2, rb_accs, bar_width,
                     label='Rule-based', color=COLOR_RULEBASED, edgecolor='white', linewidth=0.5)

    # подписываем значения accuracy над столбцами CLASSLA
    for bar, acc in zip(bars_cl, cl_accs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f'{acc:.0f}', ha='center', va='bottom', fontsize=7, color=COLOR_CLASSLA)

    # подписываем значения accuracy над столбцами rule-based
    for bar, acc in zip(bars_rb, rb_accs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f'{acc:.0f}', ha='center', va='bottom', fontsize=7, color=COLOR_RULEBASED)

    # подписываем POS-теги на оси X, добавляя количество токенов в скобках
    tick_labels = [f'{p}\n(n={t})' for p, t in zip(pos_labels, totals)]
    ax.set_xticks(x)
    ax.set_xticklabels(tick_labels, fontsize=8)

    # подписи осей на русском
    ax.set_xlabel('POS-тег', fontsize=11)
    ax.set_ylabel('Accuracy (%)', fontsize=11)
    ax.set_title('Accuracy лемматизации по POS-тегам (gold standard UD-MTB)', fontsize=13)

    # ограничиваем ось Y от 0 до 110, чтобы подписи не обрезались
    ax.set_ylim(0, 115)

    # добавляем горизонтальную сетку для удобства чтения
    ax.yaxis.grid(True, alpha=0.3, linestyle='-')
    ax.set_axisbelow(True)

    # легенда
    ax.legend(fontsize=10, loc='upper right')

    # плотная компоновка
    fig.tight_layout()

    # сохраняем картинку
    out_path = os.path.join(VIS_DIR, 'step8_accuracy_by_pos.png')
    fig.savefig(out_path, dpi=DPI)
    plt.close(fig)
    print(f'8.1 сохранено: {out_path}')


def plot_error_pie(metrics):
    """
    8.2: круговая диаграмма категорий ошибок rule-based лемматизатора
    5 категорий из error_analysis.rb_error_categories
    """
    # вытаскиваем категории ошибок
    cats = metrics['error_analysis']['rb_error_categories']

    # русские подписи для каждой категории
    labels_map = {
        'oov_rules': 'Слово вне правил (OOV)',
        'verb': 'Глагольные формы',
        'article': 'Артикль не снят',
        'irregular': 'Неправильные формы',
        'adj': 'Прилагательные',
    }

    # сортируем по количеству ошибок (убывание), чтобы самый большой сегмент был первым
    sorted_cats = sorted(cats.items(), key=lambda x: -x[1])

    # подписи и значения
    labels = [labels_map.get(k, k) for k, v in sorted_cats]
    sizes = [v for k, v in sorted_cats]
    total = sum(sizes)

    # палитра: 5 разных цветов, чтобы сегменты различались
    colors = ['#3274A1', '#E1812C', '#E15D6A', '#72B043', '#9B59B6']

    # создаём фигуру
    fig, ax = plt.subplots(figsize=(8, 6))

    # функция для формата подписей: показываем и % и количество
    def make_autopct(values):
        def autopct(pct):
            # вычисляем абсолютное значение из процента
            absolute = int(round(pct / 100.0 * sum(values)))
            return f'{pct:.1f}%\n({absolute})'
        return autopct

    # рисуем круговую диаграмму
    wedges, texts, autotexts = ax.pie(
        sizes,
        labels=labels,
        autopct=make_autopct(sizes),
        colors=colors,
        startangle=90,
        pctdistance=0.7,
        labeldistance=1.15,
        textprops={'fontsize': 10},
    )

    # делаем процентные подписи жирнее для читаемости
    for autotext in autotexts:
        autotext.set_fontsize(9)

    # заголовок на русском
    ax.set_title(f'Категории ошибок rule-based лемматизатора\n(всего {total} ошибок)',
                 fontsize=13)

    # плотная компоновка
    fig.tight_layout()

    # сохраняем
    out_path = os.path.join(VIS_DIR, 'step8_error_pie.png')
    fig.savefig(out_path, dpi=DPI)
    plt.close(fig)
    print(f'8.2 сохранено: {out_path}')


def plot_speed(metrics):
    """
    8.3: столбчатая диаграмма скорости с логарифмической шкалой Y
    CLASSLA: 113 токенов/сек, rule-based: 215963 токенов/сек
    """
    # вытаскиваем данные о скорости
    cl_speed = metrics['speed']['classla']['tokens_per_sec']
    rb_speed = metrics['speed']['rulebased']['tokens_per_sec']
    speedup = metrics['speed']['speedup']

    # названия и значения
    names = ['CLASSLA', 'Rule-based']
    speeds = [cl_speed, rb_speed]
    colors = [COLOR_CLASSLA, COLOR_RULEBASED]

    # создаём фигуру
    fig, ax = plt.subplots(figsize=(7, 5))

    # позиции столбцов
    x = np.arange(len(names))
    bar_width = 0.5

    # рисуем столбцы
    bars = ax.bar(x, speeds, bar_width, color=colors, edgecolor='white', linewidth=0.5)

    # подписываем конкретные значения на столбцах
    for bar, speed in zip(bars, speeds):
        # форматируем число с разделителем тысяч
        label = f'{speed:,.0f}'
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.3,
                label, ha='center', va='bottom', fontsize=11, fontweight='bold')

    # добавляем текст про ускорение между столбцами
    mid_x = (x[0] + x[1]) / 2
    mid_y = (cl_speed * rb_speed) ** 0.5
    ax.annotate(f'{speedup:.0f}x ускорение',
                xy=(mid_x, mid_y), fontsize=13, fontweight='bold',
                ha='center', va='center', color='#333333',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFFFCC', edgecolor='#999999', alpha=0.9))

    # логарифмическая шкала Y
    ax.set_yscale('log')

    # подписи
    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=12)
    ax.set_ylabel('Токенов в секунду (log)', fontsize=11)
    ax.set_title('Скорость лемматизации: CLASSLA vs Rule-based', fontsize=13)

    # горизонтальная сетка
    ax.yaxis.grid(True, alpha=0.3, linestyle='-')
    ax.set_axisbelow(True)

    # убираем верхнюю и правую рамку для чистоты
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # плотная компоновка
    fig.tight_layout()

    # сохраняем
    out_path = os.path.join(VIS_DIR, 'step8_speed.png')
    fig.savefig(out_path, dpi=DPI)
    plt.close(fig)
    print(f'8.3 сохранено: {out_path}')


def build_error_examples():
    """
    8.4: прогоняем gold standard через оба лемматизатора,
    находим примеры расхождений, выбираем 15 разнообразных,
    сохраняем в CSV
    """
    # импортируем pandas для работы с таблицами
    import pandas as pd

    # импортируем функции из предыдущих шагов
    from step7_1_3_evaluation import load_gold_standard, run_classla_on_gold, run_rulebased_on_gold

    # загружаем gold standard из базы
    print('загружаем gold standard для таблицы ошибок...')
    gold_rows, sentences = load_gold_standard(DB_PATH)
    print(f'  токенов: {len(gold_rows)}, предложений: {len(sentences)}')

    # прогоняем через CLASSLA
    classla_lemmas = run_classla_on_gold(sentences)
    print(f'  CLASSLA обработала {len(classla_lemmas)} токенов')

    # прогоняем через rule-based
    rb_lemmas = run_rulebased_on_gold(gold_rows)
    print(f'  rule-based обработал {len(rb_lemmas)} токенов')

    # собираем полную таблицу с результатами обоих лемматизаторов
    rows = []
    for row in gold_rows:
        key = (row['sentence_id'], row['token_id'])
        cl_lemma = classla_lemmas.get(key, '')
        rb_lemma = rb_lemmas.get(key, '')
        gold = row['lemma_gold']

        # сравниваем case-insensitive
        cl_correct = (cl_lemma.lower() == gold.lower()) if cl_lemma else False
        rb_correct = (rb_lemma.lower() == gold.lower()) if rb_lemma else False

        rows.append({
            'form': row['form'],
            'upos': row['upos'],
            'lemma_gold': gold,
            'lemma_classla': cl_lemma,
            'lemma_rulebased': rb_lemma,
            'correct_classla': cl_correct,
            'correct_rulebased': rb_correct,
        })

    # превращаем в DataFrame
    df = pd.DataFrame(rows)

    # фильтруем: хотя бы один лемматизатор ошибся
    errors_df = df[~(df['correct_classla'] & df['correct_rulebased'])].copy()
    print(f'  всего токенов с ошибками: {len(errors_df)}')

    # для разнообразия выберем по несколько примеров из разных категорий:
    # 1) rule-based ошибся, CLASSLA правильно (разные POS)
    # 2) CLASSLA ошибся, rule-based правильно
    # 3) оба ошиблись

    selected = []
    seen_forms = set()

    # категория 1: rb wrong, cl right — берём по одному из разных POS
    cat1 = errors_df[errors_df['correct_classla'] & ~errors_df['correct_rulebased']]
    for upos in ['VERB', 'NOUN', 'ADJ', 'PRON', 'DET', 'NUM']:
        # берём первый пример с этим POS, которого ещё нет в выборке
        subset = cat1[cat1['upos'] == upos]
        for _, row in subset.iterrows():
            if row['form'] not in seen_forms:
                selected.append(row)
                seen_forms.add(row['form'])
                break

    # категория 2: cl wrong, rb right — берём до 3 примеров
    cat2 = errors_df[~errors_df['correct_classla'] & errors_df['correct_rulebased']]
    count_cat2 = 0
    for _, row in cat2.iterrows():
        if row['form'] not in seen_forms and count_cat2 < 3:
            selected.append(row)
            seen_forms.add(row['form'])
            count_cat2 += 1

    # категория 3: оба ошиблись — берём до 6 примеров (разные POS)
    cat3 = errors_df[~errors_df['correct_classla'] & ~errors_df['correct_rulebased']]
    count_cat3 = 0
    for upos in ['PRON', 'VERB', 'NOUN', 'ADJ', 'DET', 'ADV', 'AUX', 'NUM']:
        subset = cat3[cat3['upos'] == upos]
        for _, row in subset.iterrows():
            if row['form'] not in seen_forms and count_cat3 < 6:
                selected.append(row)
                seen_forms.add(row['form'])
                count_cat3 += 1
                break

    # если ещё не набрали 15, добираем из оставшихся ошибок
    if len(selected) < 15:
        for _, row in errors_df.iterrows():
            if row['form'] not in seen_forms:
                selected.append(row)
                seen_forms.add(row['form'])
            if len(selected) >= 15:
                break

    # собираем финальный DataFrame
    examples_df = pd.DataFrame(selected).reset_index(drop=True)

    # сохраняем в CSV
    csv_path = os.path.join(DATASETS_DIR, 'error_examples.csv')
    examples_df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f'8.4 сохранено: {csv_path} ({len(examples_df)} строк)')

    # печатаем таблицу в stdout
    print()
    print('Таблица примеров ошибок лемматизации (15 разнообразных примеров):')
    print()

    # заголовок таблицы
    header = (f'  {"form":<16s} {"upos":<6s} {"gold":<14s} '
              f'{"classla":<14s} {"rulebased":<14s} {"cl_ok":>5s} {"rb_ok":>5s}')
    print(header)
    # разделитель
    print(f'  {"-"*16} {"-"*6} {"-"*14} {"-"*14} {"-"*14} {"-"*5} {"-"*5}')

    # строки
    for _, row in examples_df.iterrows():
        cl_mark = 'v' if row['correct_classla'] else 'x'
        rb_mark = 'v' if row['correct_rulebased'] else 'x'
        print(f'  {row["form"]:<16s} {row["upos"]:<6s} {row["lemma_gold"]:<14s} '
              f'{row["lemma_classla"]:<14s} {row["lemma_rulebased"]:<14s} '
              f'{cl_mark:>5s} {rb_mark:>5s}')

    print()

    return examples_df


def main():
    print('Шаг 8: визуализация сравнения лемматизаторов')
    print()

    # создаём папку для визуализаций, если её нет
    os.makedirs(VIS_DIR, exist_ok=True)

    # подбираем кириллический шрифт
    setup_font()
    print()

    # загружаем метрики из JSON
    metrics = load_metrics()
    print()

    # 8.1: accuracy по POS
    print('8.1: столбчатая диаграмма accuracy по POS-тегам...')
    plot_accuracy_by_pos(metrics)
    print()

    # 8.2: круговая диаграмма ошибок
    print('8.2: круговая диаграмма ошибок rule-based...')
    plot_error_pie(metrics)
    print()

    # 8.3: скорость
    print('8.3: столбчатая диаграмма скорости...')
    plot_speed(metrics)
    print()

    # 8.4: таблица примеров ошибок (прогоняем оба лемматизатора на gold standard)
    print('8.4: таблица примеров ошибок...')
    build_error_examples()
    print()

    print('Все визуализации сохранены.')

In [ ]:
# загружаем метрики и строим графики
metrics_path = r'temp/step7_1_3_metrics.json'
with open(metrics_path, 'r', encoding='utf-8') as f:
    metrics = json.load(f)

VIS_DIR = r'temp/visualization'
DPI = 150

# 8.1: accuracy по POS
plot_accuracy_by_pos(metrics)

# 8.2: круговая диаграмма ошибок
plot_error_pie(metrics)

# 8.3: скорость
plot_speed(metrics)

print("\nВсе графики сохранены в temp/visualization/")

## 2.10. Dependency parsing через spaCy

CLASSLA **не поддерживает** dependency parsing для македонского (нет обученной модели). Из доступных инструментов, **spaCy** с моделью `mk_core_news_lg` — единственный вариант с готовым dependency parsing для mk.

**Обзор инструментов:**
- **CLASSLA** — нет depparse для mk
- **TreeTagger** — нет модели для mk
- **MaltParser** — не поддерживается с ~2015 года, нет модели для mk
- **spaCy** — есть mk_core_news_lg с dependency parsing
- **UDPipe/Stanza** — слишком мало тренировочных данных (UD-MTB ~1360 токенов)

In [ ]:
# установка spaCy и скачивание модели (выполнить один раз)
# pip install spacy
# python -m spacy download mk_core_news_lg

import spacy

# загружаем mk-модель для dependency parsing
print("Загружаем spaCy модель mk_core_news_lg...")
try:
    nlp_spacy = spacy.load('mk_core_news_lg')
    print(f"Модель загружена: mk_core_news_lg")
    print(f"Версия spaCy: {spacy.__version__}")
except OSError:
    print("Модель mk_core_news_lg не найдена.")
    print("Установите: python -m spacy download mk_core_news_lg")
    nlp_spacy = None

In [ ]:
# тестируем dependency parsing на примерах
if nlp_spacy is not None:
    test_sentences = [
        "Девојката му напиша писмо на својот пријател.",
        "Мислам дека ќе врне дожд утре.",
        "Тој сакаше да прочита интересна книга."
    ]

    for sent_text in test_sentences:
        print(f'Предложение: "{sent_text}"')
        doc = nlp_spacy(sent_text)
        print(f"  {'Token':<15} {'Lemma':<15} {'POS':<8} {'Dep':<12} {'Head'}")
        print(f"  {'-'*15} {'-'*15} {'-'*8} {'-'*12} {'-'*15}")
        for token in doc:
            print(f"  {token.text:<15} {token.lemma_:<15} {token.pos_:<8} "
                  f"{token.dep_:<12} {token.head.text}")
        print()

In [ ]:
# гибридный pipeline: CLASSLA (POS, lemma) + spaCy (dependency parsing)
# функция для объединения результатов в формат CoNLL-U

def classla_to_spacy_depparse(classla_doc, spacy_nlp):
    """
    объединяет результаты CLASSLA (POS, lemma) и spaCy (dependency parsing)
    в единый формат: список предложений, каждое -- список словарей
    с полями id, form, lemma, upos, dep_rel, dep_head
    """
    all_results = []

    for sentence in classla_doc.sentences:
        # извлекаем токены, леммы и POS из CLASSLA
        words = [word.text for word in sentence.words]
        lemmas = [word.lemma for word in sentence.words]
        upos_tags = [word.upos for word in sentence.words]

        # прогоняем через spaCy для dependency parsing
        sent_text = ' '.join(words)
        spacy_doc = spacy_nlp(sent_text)

        # собираем dependency info из spaCy
        spacy_deps = []
        for token in spacy_doc:
            spacy_deps.append({
                'dep': token.dep_,
                'head_idx': token.head.i,
                'idx': token.i
            })

        # объединяем результаты
        sent_result = []
        min_len = min(len(words), len(spacy_deps))
        for i in range(min_len):
            spacy_head = spacy_deps[i]['head_idx'] + 1
            if spacy_deps[i]['idx'] == spacy_deps[i]['head_idx']:
                spacy_head = 0  # корень

            sent_result.append({
                'id': i + 1,
                'form': words[i],
                'lemma': lemmas[i],
                'upos': upos_tags[i],
                'dep_rel': spacy_deps[i]['dep'],
                'dep_head': spacy_head
            })

        all_results.append(sent_result)

    return all_results


def hybrid_to_conll(tokens):
    """конвертирует список токенов в формат CoNLL-U (10 столбцов)"""
    lines = []
    for tok in tokens:
        line = '\t'.join([
            str(tok['id']), tok['form'], tok['lemma'], tok['upos'],
            '_', '_', str(tok['dep_head']), tok['dep_rel'], '_', '_'
        ])
        lines.append(line)
    return '\n'.join(lines)


# демонстрация гибридного pipeline
if nlp_spacy is not None:
    print("Гибридный pipeline: CLASSLA + spaCy")
    print()

    test_text = "Девојката му напиша писмо на својот пријател ."
    # CLASSLA: tokenize + POS + lemma
    classla_doc = nlp([test_text.strip().split()][0]
                      if False else test_text)

    # объединяем с spaCy dep parsing
    results = classla_to_spacy_depparse(classla_doc, nlp_spacy)

    for sent_tokens in results:
        print(f"  {'ID':<4} {'Form':<18} {'Lemma':<18} {'UPOS':<8} {'DepRel':<12} {'Head'}")
        print(f"  {'-'*4} {'-'*18} {'-'*18} {'-'*8} {'-'*12} {'-'*5}")
        for tok in sent_tokens:
            print(f"  {tok['id']:<4} {tok['form']:<18} {tok['lemma']:<18} "
                  f"{tok['upos']:<8} {tok['dep_rel']:<12} {tok['dep_head']}")
        print()
        print("  CoNLL-U формат:")
        print(hybrid_to_conll(sent_tokens))

### Итоговый набор NLP-инструментов

| Задача | Инструмент | Обоснование |
|--------|-----------|-------------|
| Токенизация и POS-тегирование | CLASSLA 2.2.1 | Лучшие модели для южнославянских языков |
| Лемматизация (основная) | CLASSLA | Accuracy 85.9% на gold standard |
| Лемматизация (сравнение) | Собственный rule-based | Для демонстрации разницы подходов |
| Dependency parsing | spaCy (`mk_core_news_lg`) | Единственный инструмент с готовой моделью для mk |

**Гибридный подход:** CLASSLA для POS/lemma, spaCy для dependency parsing. Функция `classla_to_spacy_depparse` объединяет результаты в единый формат CoNLL-U.

## 2.11. Выводы

### Результаты сравнения лемматизаторов

На gold standard UD Macedonian-MTB (1360 токенов, 155 предложений):

| Метрика | CLASSLA | Rule-based |
|---------|---------|------------|
| Accuracy (case-insensitive) | **85.9%** | 73.6% |
| Accuracy (exact match) | **84.8%** | 72.5% |
| Accuracy (NOUN) | **98.3%** | 68.7% |
| Accuracy (VERB) | **82.3%** | 59.1% |
| Accuracy (ADJ) | **90.9%** | 50.0% |
| Скорость (токенов/сек) | 113 | **215,963** |
| Ускорение | - | **1,911x** |

### Основные выводы

1. **CLASSLA значительно точнее** rule-based лемматизатора (+12.3 п.п. accuracy). Нейросеть учитывает контекст, что критично для омонимичных форм.

2. **Rule-based в 1911 раз быстрее**, но для корпуса в 4.6M слов разница между ~50 мин (CLASSLA) и ~33 сек (rule-based) не критична.

3. **Главные источники ошибок rule-based:** слова вне правил (165 ошибок), глагольные формы (115), артикли (35), неправильные формы (22), прилагательные (22).

4. **spaCy с моделью mk_core_news_lg** — единственный доступный инструмент для dependency parsing на македонском. Гибридный подход (CLASSLA + spaCy) позволяет получить полный набор NLP-признаков: токенизацию, POS, леммы и зависимости.

### Созданные датасеты

| Файл | Содержание |
|------|-----------|
| `datasets/nlp_data.db` | SQLite-база: classla_tokens, rulebased_tokens, gold_standard, processing_log |
| `datasets/lemmatizer_comparison.csv` | Сводная таблица сравнения двух лемматизаторов |
| `datasets/error_examples.csv` | 15 примеров ошибок для наглядного сравнения |
| `datasets/gold_standard_lemmas.csv` | Gold standard (UD-MTB) в CSV-формате |